### Libraries

In [ ]:
# Colin, Kostas, Ellie, Heejin, Fiona
# Standard library
import math
import re
from datetime import datetime, timedelta
from pathlib import Path

# Data processing
import numpy as np
import pandas as pd
import xarray as xr

# Geospatial
import geopandas as gpd
from shapely import wkt
from shapely.geometry import Point
from scipy.spatial import cKDTree
import rasterio
from rasterio.plot import show
import earthaccess

# Visualization
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

### Money Figure

In [ ]:
### import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch, Polygon as MplPolygon, Circle
from matplotlib.ticker import FuncFormatter
import geopandas as gpd
import contextily as cx
from pathlib import Path
from shapely.geometry import box
import rasterio

# 1. Reach 로드 (ISA 0.06)
selected_12 = gpd.read_file('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/selected_12_vD.shp')
fctimp_str = '0.06'
sample = selected_12[np.isclose(selected_12["fctimp_mea"], float(fctimp_str), atol=0.01)].copy()

# 4326 좌표
sample_4326 = sample.to_crs('EPSG:4326')
bounds_4326 = sample_4326.total_bounds

# 양쪽으로 넓히기
lon_buffer = 0.1
lat_buffer = 0.005

expanded_bounds_4326 = [
    bounds_4326[0] - lon_buffer,
    bounds_4326[1] - lat_buffer,
    bounds_4326[2] + lon_buffer,
    bounds_4326[3] + lat_buffer
]

# 3857로 변환
expanded_box = gpd.GeoDataFrame(
    geometry=[box(*expanded_bounds_4326)], 
    crs='EPSG:4326'
).to_crs('EPSG:3857')
expanded_bounds_3857 = expanded_box.total_bounds
sample_3857 = sample.to_crs('EPSG:3857')

# 2. NLCD 데이터 로드
nlcd_path = '/work/pi_kandread_umass_edu/swot-urban/0_data_collection/1_nlcd/Annual_NLCD_LndCov_2024_CU_C1V1_EPSG4326.tif'

with rasterio.open(nlcd_path) as src:
    window = src.window(*expanded_bounds_4326)
    nlcd_data = src.read(1, window=window)
    nlcd_bounds = rasterio.windows.bounds(window, src.transform)

# NLCD bounds를 3857로 변환
nlcd_box_3857 = gpd.GeoDataFrame(
    geometry=[box(nlcd_bounds[0], nlcd_bounds[1], nlcd_bounds[2], nlcd_bounds[3])],
    crs='EPSG:4326'
).to_crs('EPSG:3857')
nlcd_extent_3857 = nlcd_box_3857.total_bounds

# 3. NLCD 오버레이 이미지 생성 (RGBA)
nlcd_overlay = np.zeros((*nlcd_data.shape, 4), dtype=np.float32)

developed_mask = np.isin(nlcd_data, [22, 23, 24])
nlcd_overlay[developed_mask] = [1.0, 0.0, 0.0, 0.3]

water_mask = nlcd_data == 11
nlcd_overlay[water_mask] = [0.0, 0.4, 1.0, 0.7]

other_mask = (nlcd_data > 0) & ~developed_mask & ~water_mask
nlcd_overlay[other_mask] = [0.5, 0.5, 0.5, 0.7]

# 4. 3857 좌표를 4326으로 변환하는 함수 (축 라벨용)
from pyproj import Transformer

transformer_to_4326 = Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)

def lon_formatter_3857(x, pos):
    lon, lat = transformer_to_4326.transform(x, 0)
    if lon < 0:
        return f'{abs(lon):.2f}°W'
    else:
        return f'{lon:.2f}°E'

def lat_formatter_3857(y, pos):
    lon, lat = transformer_to_4326.transform(0, y)
    if lat > 0:
        return f'{lat:.2f}°N'
    else:
        return f'{abs(lat):.2f}°S'


# 5. 위성 궤도 그리기 함수
def draw_satellite_path(ax, x_min, x_max, y_min, y_max, heading, cycle_id, pass_id, tile_id, 
                        position='left', color='#303030', alpha=0.8):
    going_north = (0 <= heading < 90) or (270 <= heading < 360)
    
    x_span = x_max - x_min
    y_span = y_max - y_min
    band_width = x_span * 0.06
    
    if position == 'left':
        center_x = x_min + x_span * 0.15
    else:
        center_x = x_max - x_span * 0.15
    
    x_offset = y_span * 0.08
    
    if going_north:
        points = [
            (center_x - band_width/2 - x_offset, y_min),
            (center_x + band_width/2 - x_offset, y_min),
            (center_x + band_width/2 + x_offset, y_max),
            (center_x - band_width/2 + x_offset, y_max),
        ]
    else:
        points = [
            (center_x - band_width/2 + x_offset, y_min),
            (center_x + band_width/2 + x_offset, y_min),
            (center_x + band_width/2 - x_offset, y_max),
            (center_x - band_width/2 - x_offset, y_max),
        ]
    
    band_patch = MplPolygon(points, facecolor=color, edgecolor='black', 
                             alpha=alpha, linewidth=1.5, zorder=6)
    ax.add_patch(band_patch)
    
    # 텍스트 회전 (경로 방향)
    if going_north:
        rotation = 76
    else:
        rotation = -76
    
    text_y_center = (y_min + y_max) / 2
    text_spacing = y_span * 0.10
    text_size = 10
    
    # 위성 위치 (흰색 원)
    if going_north:
        sat_x = center_x + x_offset * 0.5
        sat_y = text_y_center + y_span * 0.25
    else:
        sat_x = center_x - x_offset * 0.5
        sat_y = text_y_center + y_span * 0.25
    
    satellite_circle = Circle((sat_x, sat_y), radius=x_span * 0.0012, 
                               facecolor='white', edgecolor='black', 
                               linewidth=2, zorder=8)
    ax.add_patch(satellite_circle)
    
    # ax.text(sat_x, sat_y - y_span * 0.06, 'Path of\nSatellite', 
    #         fontsize=text_size, fontweight='bold', color='white', 
    #         ha='center', va='top', rotation=rotation, zorder=7)
    
    # info_y = text_y_center - y_span * 0.15
    # ax.text(center_x, info_y, f'Tile: {tile_id}', 
    #         fontsize=text_size, fontweight='bold', color='white', 
    #         ha='center', va='center', rotation=rotation, zorder=7)
    # ax.text(center_x, info_y - text_spacing * 1.2, f'Pass: {pass_id}', 
    #         fontsize=text_size, fontweight='bold', color='white', 
    #         ha='center', va='center', rotation=rotation, zorder=7)
    # ax.text(center_x, info_y - text_spacing * 2.4, f'Cycle: {cycle_id}', 
    #         fontsize=text_size, fontweight='bold', color='white', 
    #         ha='center', va='center', rotation=rotation, zorder=7)
    
    # if going_north:
    #     ax.text(center_x + x_offset, y_max - y_span * 0.03, f'{heading:.0f}°', 
    #             fontsize=text_size + 2, fontweight='bold', color='white', 
    #             ha='center', va='top', rotation=rotation, zorder=7)
    # else:
    #     ax.text(center_x - x_offset, y_min + y_span * 0.03, f'{heading:.0f}°', 
    #             fontsize=text_size + 2, fontweight='bold', color='white', 
    #             ha='center', va='bottom', rotation=rotation, zorder=7)


# 6. 플롯
fig, ax = plt.subplots(figsize=(18, 14))

ax.set_xlim(expanded_bounds_3857[0], expanded_bounds_3857[2])
ax.set_ylim(expanded_bounds_3857[1], expanded_bounds_3857[3])

# 1) ESRI World Imagery 배경
cx.add_basemap(ax, source=cx.providers.Esri.WorldImagery, zoom=16)

# 2) NLCD 오버레이
ax.imshow(nlcd_overlay, 
          extent=[nlcd_extent_3857[0], nlcd_extent_3857[2], nlcd_extent_3857[1], nlcd_extent_3857[3]],
          origin='upper', zorder=2, interpolation='nearest')

# 3) Reach 경계
sample_3857.boundary.plot(ax=ax, color='black', linewidth=4, zorder=5)

# 4) 위성 궤도 두 개
draw_satellite_path(ax, expanded_bounds_3857[0], expanded_bounds_3857[2],
                    expanded_bounds_3857[1], expanded_bounds_3857[3],
                    heading=166, cycle_id='039', pass_id='272', tile_id='082R',
                    position='right', color='teal', alpha=0.5)

draw_satellite_path(ax, expanded_bounds_3857[0], expanded_bounds_3857[2],
                    expanded_bounds_3857[1], expanded_bounds_3857[3],
                    heading=14, cycle_id='039', pass_id='315', tile_id='227R',
                    position='left', color='coral', alpha=0.5)

# 5) 축 라벨 (E/W/N/S 형식)
ax.xaxis.set_major_formatter(FuncFormatter(lon_formatter_3857))
ax.yaxis.set_major_formatter(FuncFormatter(lat_formatter_3857))
ax.tick_params(axis='both', labelsize=12)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=15, ha='right')

ax.set_xlabel('Longitude', fontsize=14, fontweight='bold')
ax.set_ylabel('Latitude', fontsize=14, fontweight='bold')

# ax.set_title(f'ISA = {fctimp_str} | Urban (Red) | Water (Blue) | Other (Gray)', 
#              fontsize=16, fontweight='bold')

# 범례
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, FancyArrowPatch

legend_elements = [
    Line2D([0], [0], color='coral', linewidth=3, marker='>', label='Measurement Path'),
    Line2D([0], [0], color='teal', linewidth=3, marker='>', label='Measurement Path'),
    Patch(facecolor='none', edgecolor='cyan', linewidth=2, label='Reach Buffer Zone'),
    Line2D([0], [0], color='black', linewidth=2, marker='>', markersize=8,
       label='Satellite Path'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='white', 
       markeredgecolor='black', markersize=10, label='SWOT Node'),
    Patch(facecolor='red', alpha=0.35, edgecolor='black', label='Developed (ISA)'),
    Patch(facecolor='blue', alpha=0.4, edgecolor='black', label='Open Water'),
    Patch(facecolor='gray', alpha=0.5, edgecolor='black', label='Other Land Cover'),
]

ax.legend(handles=legend_elements, loc='lower center', fontsize=14, bbox_to_anchor=(0.3, 0.05))

plt.savefig('figures/money_figure_with_paths.png', dpi=600, bbox_inches='tight')
plt.show()
print("✓ Saved: figures/money_figure_with_paths.png")

### PIXC

In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
import earthaccess
import re
from IPython.display import clear_output
import numpy as np

def filter_best_granules(results):
    """Keep best version: PGD0 > PID0, highest ProductCounter"""
    granule_dict = {}
    
    for r in results:
        url = r.data_links()[0]
        filename = url.split('/')[-1]
        
        # Parse: SWOT_L2_HR_PIXC_040_315_228R_20251023T044410_20251023T044421_PID0_03.nc
        match = re.match(r'(.+)_(P[IG]D\d)_(\d+)\.nc', filename)
        if not match:
            continue
        
        base = match.group(1)
        processor = match.group(2)  # PID0 or PGD0
        counter = int(match.group(3))
        
        if base not in granule_dict:
            granule_dict[base] = (r, processor, counter)
        else:
            _, old_proc, old_cnt = granule_dict[base]
            
            # PGD0 > PID0
            proc_better = (processor == 'PGD0' and old_proc == 'PID0')
            proc_same = (processor == old_proc)
            cnt_better = (counter > old_cnt)
            
            if proc_better or (proc_same and cnt_better):
                granule_dict[base] = (r, processor, counter)
    
    return [r for r, _, _ in granule_dict.values()]


selected_12 = gpd.read_parquet('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/all_reaches_vD.parquet')
data_dir = Path('/work/pi_kandread_umass_edu/swot-urban/5_fctimp/')
parquet_files = sorted(data_dir.glob('fctimp_mean_*.parquet'))
print(f"Found {len(parquet_files)} parquet files\n")

earthaccess.login()

for i, pq_file in enumerate(parquet_files, 1):
    clear_output(wait=True)
    print(f"Processing {i}/{len(parquet_files)}: {pq_file.name}")
    
    reach_id_str = pq_file.stem.split('_')[-1]
    
    try:
        reach_id_val = int(reach_id_str)
    except ValueError:
        reach_id_val = reach_id_str
    
    mask = selected_12["reach_id"] == reach_id_val
    sample = selected_12.loc[mask].copy()
    
    if sample.empty:
        print(f"fctimp {reach_id_str}: No matching reach, skipping")
        continue
    
    bounds = sample.total_bounds
    bbox = (bounds[0] - 0.1, bounds[1] - 0.1, bounds[2] + 0.1, bounds[3] + 0.1)
    
    df = pd.read_parquet(pq_file)
    
    dates = pd.to_datetime(df['time_str']).dt.date
    if len(dates) == 0: continue
    
    min_date = dates.min().strftime('%Y-%m-%d')
    max_date = dates.max().strftime('%Y-%m-%d')

    print(f"reach {reach_id_str}: {len(set(dates))} dates ({min_date} ~ {max_date})")
    
    all_results = earthaccess.search_data(
        short_name='SWOT_L2_HR_PIXC_D',
        bounding_box=bbox,
        temporal=(min_date, max_date)
    )
    
    # Filter to best versions
    filtered = filter_best_granules(all_results)
    
    outfile = f'PIXC_url_all/swot_PIXC_vD_fctimp_{reach_id_str}_urls.txt'
    output_dir = f'PIXC_url_all'
    Path(output_dir).mkdir(exist_ok=True)
    
        for r in filtered:
            f.write(r.data_links()[0] + '\n')
    
    print(f"  → {len(all_results)} → {len(filtered)} granules → {outfile}\n")

In [ ]:
### tvp는 streaming으로 다운받아서 parquet으로 저장!! -->swot_tvp_download.py and run_swot.sh

In [ ]:
import earthaccess
import h5py
import pandas as pd
from pathlib import Path
import numpy as np

earthaccess.login()

# 경로 설정
url_dir = Path("PIXC_url_all")
parquet_dir = Path("/work/pi_kandread_umass_edu/swot-urban/6_pixc_tvp/SWOT_TVP_Data")
download_dir = Path("/work/pi_kandread_umass_edu/swot-urban/PIXC_test_download")
download_dir.mkdir(exist_ok=True)

# 첫 번째 reach의 URL 5개만
txt_files = sorted([f for f in url_dir.glob("*_urls.txt")])
reach_file = txt_files[0]
reach_id = reach_file.stem.split('_')[-2]

with open(reach_file, 'r') as f:
    urls = [line.strip() for line in f if line.strip()][:5]

print(f"Reach: {reach_id}")
print(f"테스트 URL 5개:\n")

# 1. PIXC 5개 다운로드
print("=" * 50)
print("1. PIXC 다운로드 중...")
downloaded = earthaccess.download(urls, download_dir)
print(f"   다운로드 완료: {len(downloaded)}개\n")

# 2. 원본 PIXC에서 tvp 읽기
print("=" * 50)
print("2. 원본 PIXC에서 tvp 읽기...")
original_dfs = []
for nc_file in sorted(download_dir.glob("*.nc")):
    with h5py.File(nc_file, 'r') as f:
        tvp = f['tvp']
        df = pd.DataFrame({
            'time': tvp['time'][:],
            'latitude': tvp['latitude'][:],
            'longitude': tvp['longitude'][:],
            'altitude': tvp['altitude'][:]
        })
        df['source_file'] = nc_file.name
        original_dfs.append(df)
        print(f"   {nc_file.name}: {len(df)} rows")

original_df = pd.concat(original_dfs, ignore_index=True)
print(f"   원본 총: {len(original_df)} rows\n")

# 3. Parquet에서 해당 파일들만 읽기
print("=" * 50)
print("3. Parquet에서 읽기...")
parquet_path = parquet_dir / f"tvp_reach_{reach_id}.parquet"
parquet_df = pd.read_parquet(parquet_path)

# 다운받은 5개 파일만 필터링
source_files = [url.split('/')[-1] for url in urls]
parquet_subset = parquet_df[parquet_df['source_file'].isin(source_files)].reset_index(drop=True)
print(f"   Parquet 총: {len(parquet_df)} rows")
print(f"   필터링 후: {len(parquet_subset)} rows\n")

# 4. 비교
print("=" * 50)
print("4. 데이터 비교...")

# 같은 순서로 정렬
original_sorted = original_df.sort_values(['source_file', 'time']).reset_index(drop=True)
parquet_sorted = parquet_subset.sort_values(['source_file', 'time']).reset_index(drop=True)

# 비교
match_rows = len(original_sorted) == len(parquet_sorted)
match_time = np.allclose(original_sorted['time'], parquet_sorted['time'], equal_nan=True)
match_lat = np.allclose(original_sorted['latitude'], parquet_sorted['latitude'], equal_nan=True)
match_lon = np.allclose(original_sorted['longitude'], parquet_sorted['longitude'], equal_nan=True)
match_alt = np.allclose(original_sorted['altitude'], parquet_sorted['altitude'], equal_nan=True)

print(f"   행 개수 일치: {match_rows} ({len(original_sorted)} vs {len(parquet_sorted)})")
print(f"   time 일치: {match_time}")
print(f"   latitude 일치: {match_lat}")
print(f"   longitude 일치: {match_lon}")
print(f"   altitude 일치: {match_alt}")

print("\n" + "=" * 50)
if all([match_rows, match_time, match_lat, match_lon, match_alt]):
    print("✅ 검증 성공! 데이터가 일치합니다.")
else:
    print("❌ 검증 실패! 데이터가 다릅니다.")

# 샘플 출력
print("\n원본 샘플:")
print(original_sorted.head())
print("\nParquet 샘플:")
print(parquet_sorted.head())

### PIXCVec

In [ ]:
def filter_best_granules(results):
    """Keep best version: PGD0 > PID0, highest ProductCounter"""
    granule_dict = {}
    
    for r in results:
        url = r.data_links()[0]
        filename = url.split('/')[-1]
        
        # Parse: SWOT_L2_HR_PIXCVec_040_315_228R_20251023T044410_20251023T044421_PID0_03.nc
        match = re.match(r'(.+)_(P[IG]D\d)_(\d+)\.nc', filename)
        if not match:
            continue
        
        base = match.group(1)
        processor = match.group(2)  # PID0 or PGD0
        counter = int(match.group(3))
        
        if base not in granule_dict:
            granule_dict[base] = (r, processor, counter)
        else:
            _, old_proc, old_cnt = granule_dict[base]
            
            # PGD0 > PID0
            proc_better = (processor == 'PGD0' and old_proc == 'PID0')
            proc_same = (processor == old_proc)
            cnt_better = (counter > old_cnt)
            
            if proc_better or (proc_same and cnt_better):
                granule_dict[base] = (r, processor, counter)
    
    return [r for r, _, _ in granule_dict.values()]


selected_12 = gpd.read_parquet('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/all_reaches_vD.parquet')
data_dir = Path('/work/pi_kandread_umass_edu/swot-urban/5_fctimp/')
parquet_files = sorted(data_dir.glob('fctimp_mean_*.parquet'))
print(f"Found {len(parquet_files)} parquet files\n")

earthaccess.login()

for i, pq_file in enumerate(parquet_files, 1):
    clear_output(wait=True)
    print(f"Processing {i}/{len(parquet_files)}: {pq_file.name}")
    
    reach_id_str = pq_file.stem.split('_')[-1]
    
    try:
        reach_id_val = int(reach_id_str)
    except ValueError:
        reach_id_val = reach_id_str
    
    mask = selected_12["reach_id"] == reach_id_val
    sample = selected_12.loc[mask].copy()
    
    if sample.empty:
        print(f"reach {reach_id_str}: No matching reach, skipping")
        continue
        
    bounds = sample.total_bounds
    bbox = (bounds[0] - 0.1, bounds[1] - 0.1, bounds[2] + 0.1, bounds[3] + 0.1)
    
    df = pd.read_parquet(pq_file)
    
    # Calculate date range (Optimization)
    dates = pd.to_datetime(df['time_str']).dt.date
    if len(dates) == 0: continue
    
    min_date = dates.min().strftime('%Y-%m-%d')
    max_date = dates.max().strftime('%Y-%m-%d')
    
    print(f"reach {reach_id_str}: {len(set(dates))} dates ({min_date} ~ {max_date})")
    
    # Single search for the whole range
    all_results = earthaccess.search_data(
        short_name='SWOT_L2_HR_PIXCVec_D',
        bounding_box=bbox,
        temporal=(min_date, max_date)
    )
    
    # Filter to best versions
    filtered = filter_best_granules(all_results)
    
    outfile = f'PIXCVec_url_all/swot_PIXCVec_vD_fctimp_{reach_id_str}_urls.txt'
    output_dir = f'PIXCVec_url_all'
    Path(output_dir).mkdir(exist_ok=True)
    
    with open(outfile, 'w') as f:
        for r in filtered:
            f.write(r.data_links()[0] + '\n')
    
    print(f"  → {len(all_results)} → {len(filtered)} granules → {outfile}\n")

### Height and Sig0 Comparison by legneds

#### Overall comparison

In [ ]:
def filter_best_pixc_files(files):
    file_dict = {}
    for f in files:
        match = re.match(r'(.+)_(P[IG]D\d)_(\d+)\.nc', f.name)
        if not match:
            continue
        base, processor, counter = match.group(1), match.group(2), int(match.group(3))
        if base not in file_dict:
            file_dict[base] = (f, processor, counter)
        else:
            _, old_proc, old_cnt = file_dict[base]
            if (processor == 'PGD0' and old_proc == 'PID0') or (processor == old_proc and counter > old_cnt):
                file_dict[base] = (f, processor, counter)
    return [f for f, _, _ in file_dict.values()]

def get_tile_from_filename(filename):
    parts = filename.split('_')
    if len(parts) >= 7:
        return parts[6]
    return 'UNK'

def match_and_group_pixc(time_str, files, crid, window_minutes=5):
    dt = pd.to_datetime(time_str)
    if dt.tzinfo:
        dt = dt.tz_localize(None)
    start, end = dt - timedelta(minutes=window_minutes), dt + timedelta(minutes=window_minutes)
    groups = {}
    for f in files:
        parts = f.name.split('_')
        if len(parts) >= 10:
            try:
                fs, fe = pd.to_datetime(parts[7], format='%Y%m%dT%H%M%S'), pd.to_datetime(parts[8], format='%Y%m%dT%H%M%S')
                if not (fe < start or fs > end) and parts[9] == crid:
                    key = f"{parts[7]}_{parts[8]}"
                    groups.setdefault(key, []).append(f)
            except:
                continue
    return groups

def find_matching_pixcvec(pixc_file, pixcvec_files):
    pixc_name = pixc_file.name.replace('PIXC_', 'PIXCVec_')
    for pvf in pixcvec_files:
        if pvf.name == pixc_name:
            return pvf
    return None

def extract_selected_pixels(pixc_file, pixcvec_file, polygon_gdf):
    """Extract ONLY selected pixels (via PIXCVec) within polygon"""
    ds = xr.open_dataset(pixc_file, group='pixel_cloud')
    lon, lat = ds['longitude'].values, ds['latitude'].values
    bounds = polygon_gdf.total_bounds
    
    bbox_mask = ((lon >= bounds[0]) & (lon <= bounds[2]) & (lat >= bounds[1]) & (lat <= bounds[3]))
    if bbox_mask.sum() == 0:
        ds.close()
        return None
    
    classification = ds['classification'].values[bbox_mask]
    water_mask = classification > 2
    if water_mask.sum() == 0:
        ds.close()
        return None
    
    lon_filt, lat_filt = lon[bbox_mask][water_mask], lat[bbox_mask][water_mask]
    poly = polygon_gdf.geometry.iloc[0]
    inside_mask = np.array([poly.contains(Point(x, y)) for x, y in zip(lon_filt, lat_filt)])
    if inside_mask.sum() == 0:
        ds.close()
        return None
    
    final_idx = np.where(bbox_mask)[0][water_mask][inside_mask]
    
    # Get PIXCVec selection
    ds_vec = xr.open_dataset(pixcvec_file)
    reach_id = ds_vec['reach_id'].values[final_idx]
    node_id = ds_vec['node_id'].values[final_idx]
    obs_id = ds_vec['obs_id'].values[final_idx]
    lake_id = ds_vec['lake_id'].values[final_idx]
    
    def is_valid(arr):
        return np.array([len(x) > 0 and x != b'' for x in arr]) if arr.dtype.kind == 'S' else arr > 0
    
    selected_mask = is_valid(reach_id) & is_valid(node_id) & ~is_valid(obs_id) & ~is_valid(lake_id)
    
    # Apply selection mask - ONLY return selected pixels
    selected_idx = final_idx[selected_mask]
    
    if len(selected_idx) == 0:
        ds.close()
        ds_vec.close()
        return None
    
    result = {
        'lon': lon[selected_idx],
        'lat': lat[selected_idx],
        'height': ds['height'].values[selected_idx],
        'classification': ds['classification'].values[selected_idx],
    }
    
    ds.close()
    ds_vec.close()
    return result

def extract_and_merge_selected_pixels(pixc_files, pixcvec_files, polygon_gdf):
    """Extract and merge selected pixels from multiple tiles"""
    all_data = []
    for pf, pvf in zip(pixc_files, pixcvec_files):
        data = extract_selected_pixels(pf, pvf, polygon_gdf)
        if data:
            all_data.append(data)
    
    if not all_data:
        return None
    if len(all_data) == 1:
        return all_data[0]
    
    merged = {k: np.concatenate([d[k] for d in all_data]) for k in all_data[0].keys()}
    return merged

def plot_height_by_classification(normal_data, spike_data, normal_info, spike_info, output_dir, time_str, fctimp_str):
    """
    Boxplot of height distribution by classification for Normal vs Spike
    Only for selected pixels with classification 3, 4, 6, 7
    """
    
    # Classification mapping
    class_map = {
        3: 'water_near_land',
        4: 'open_water',
        6: 'low_coh_water_near_land',
        7: 'open_low_coh_water'
    }
    class_ids = [3, 4, 6, 7]
    class_labels = [class_map[c] for c in class_ids]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    for ax, (data, info, label) in zip(axes, [
        (normal_data, normal_info, 'NORMAL'),
        (spike_data, spike_info, 'SPIKE')
    ]):
        color = 'green' if label == 'NORMAL' else 'red'
        
        # Filter valid heights
        valid_mask = np.isfinite(data['height'])
        heights = data['height'][valid_mask]
        classifications = data['classification'][valid_mask]
        
        # Get heights by classification
        height_by_class = []
        sample_counts = []
        for c in class_ids:
            h = heights[classifications == c]
            height_by_class.append(h)
            sample_counts.append(len(h))
        
        # Total statistics
        total_count = len(heights)
        mean_height = np.mean(heights) if total_count > 0 else np.nan
        
        # Create boxplot
        bp = ax.boxplot(height_by_class, positions=range(len(class_ids)), 
                        widths=0.6, patch_artist=True)
        
        # Style boxplot
        for box in bp['boxes']:
            box.set_facecolor(color)
            box.set_alpha(0.5)
        for whisker in bp['whiskers']:
            whisker.set_color(color)
        for cap in bp['caps']:
            cap.set_color(color)
        for median in bp['medians']:
            median.set_color('black')
            median.set_linewidth(2)
        for flier in bp['fliers']:
            flier.set_markerfacecolor(color)
            flier.set_markeredgecolor(color)
            flier.set_alpha(0.3)
        
        # Add sample count above each box
        for i, (pos, count) in enumerate(zip(range(len(class_ids)), sample_counts)):
            # Get the upper whisker position for label placement
            if count > 0:
                upper = np.percentile(height_by_class[i], 75) + 1.5 * (np.percentile(height_by_class[i], 75) - np.percentile(height_by_class[i], 25))
                upper = min(upper, np.max(height_by_class[i]))
            else:
                upper = 0
            ax.text(pos, upper + 0.5, f'n={count}', ha='center', va='bottom', 
                    fontsize=9, fontweight='bold', color='black')
        
        # Add mean height line
        if np.isfinite(mean_height):
            ax.axhline(y=mean_height, color='red', linestyle='--', linewidth=2, alpha=0.7, zorder=5)
        
        # Add total statistics text (top right)
        stats_text = f'Total: {total_count} pixels\nMean: {mean_height:.2f} m'
        ax.text(0.98, 0.98, stats_text, transform=ax.transAxes, 
                ha='right', va='top', fontsize=10, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='red', alpha=0.9))
        
        # Labels and title
        ax.set_xticks(range(len(class_ids)))
        ax.set_xticklabels(class_labels, rotation=30, ha='right', fontsize=9)
        ax.set_ylabel('Height (m)', fontsize=11)
        ax.set_xlabel('Classification', fontsize=11)
        ax.set_title(f'{label} | {info["heading"]:.0f}°-{info["tile"]}\nError: {info["error"]:.2f}m', 
                     fontsize=12, fontweight='bold', color=color)
        ax.grid(axis='y', alpha=0.3)
        
        # Set y-axis limits with some padding
        all_heights = np.concatenate([h for h in height_by_class if len(h) > 0])
        if len(all_heights) > 0:
            ymin = np.percentile(all_heights, 1) - 1
            ymax = np.percentile(all_heights, 99) + 2
            ax.set_ylim(ymin, ymax)
    
    fig.suptitle(f'ISA={fctimp_str} | Height Distribution by Classification (Selected Pixels Only)\n'
                 f'Normal: {normal_info["time_str"]} vs Spike: {spike_info["time_str"]}', 
                 fontsize=12, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.savefig(f'{output_dir}/COMPARE_{time_str}_HeightByClass.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"  Saved: {output_dir}/COMPARE_{time_str}_HeightByClass.png")

def plot_height_cdf_by_classification(normal_data, spike_data, normal_info, spike_info,
                                      output_dir, time_str, fctimp_str):
    """
    Height ECDF by classification for Normal vs Spike
    - 4개의 세로 subplot (water_near_land, open_water, low_coh_water_near_land, open_low_coh_water)
    - x축(Height[m])은 공유
    - 각 subplot 안에 Normal/Spike CDF + 각 그룹 mean 세로선
    - 추가: 전체 Normal mean, 전체 Spike mean 세로선(모든 subplot 공통)
    """

    # Classification mapping
    class_map = {
        3: 'water_near_land',
        4: 'open_water',
        6: 'low_coh_water_near_land',
        7: 'open_low_coh_water'
    }
    class_ids = [3, 4, 6, 7]

    # helper: 클래스별 height 분리
    def split_by_class(data):
        mask = np.isfinite(data['height'])
        h = data['height'][mask]
        cls = data['classification'][mask]
        out = {}
        for cid in class_ids:
            out[cid] = h[cls == cid]
        return out

    normal_by_class = split_by_class(normal_data)
    spike_by_class  = split_by_class(spike_data)

    # ---- 전체 Normal / Spike 높이 모아서 global mean 계산 ----
    norm_all_list = [v for v in normal_by_class.values() if len(v) > 0]
    spike_all_list = [v for v in spike_by_class.values() if len(v) > 0]

    global_mean_normal = np.mean(np.concatenate(norm_all_list)) if len(norm_all_list) > 0 else None
    global_mean_spike  = np.mean(np.concatenate(spike_all_list)) if len(spike_all_list) > 0 else None

    # ---- x축 범위 (모든 높이 합쳐서) ----
    all_heights = []
    for d in (normal_by_class, spike_by_class):
        for arr in d.values():
            if len(arr) > 0:
                all_heights.append(arr)
    if len(all_heights) == 0:
        print("  No valid heights to plot CDF.")
        return
    all_heights = np.concatenate(all_heights)
    xmin = np.percentile(all_heights, 1) - 0.5
    xmax = np.percentile(all_heights, 99) + 0.5

    # figure: 세로 4개, x축 공유
    fig, axes = plt.subplots(len(class_ids), 1, figsize=(14, 10), sharex=True)
    if len(class_ids) == 1:
        axes = [axes]

    for ax, cid in zip(axes, class_ids):
        label = class_map[cid]

        h_norm = normal_by_class[cid]
        h_spike = spike_by_class[cid]

        # CDF 함수
        def ecdf(a):
            a_sort = np.sort(a)
            y = np.arange(1, len(a_sort) + 1) / len(a_sort)
            return a_sort, y

        plotted_any = False
        handles = []
        labels = []

        # Normal CDF
        if len(h_norm) > 0:
            x_n, y_n = ecdf(h_norm)
            ln, = ax.step(x_n, y_n, where='post', color='green',
                          label=f'Normal (n={len(h_norm)})')
            mean_n = np.mean(h_norm)
            # per-class mean
            ax.axvline(mean_n, color='green', linestyle='--', alpha=0.7)
            handles.append(ln)
            labels.append(f'Normal (n={len(h_norm)}, μ={mean_n:.2f} m)')
            plotted_any = True

        # Spike CDF
        if len(h_spike) > 0:
            x_s, y_s = ecdf(h_spike)
            ls, = ax.step(x_s, y_s, where='post', color='red',
                          label=f'Spike (n={len(h_spike)})')
            mean_s = np.mean(h_spike)
            # per-class mean
            ax.axvline(mean_s, color='red', linestyle='--', alpha=0.7)
            handles.append(ls)
            labels.append(f'Spike (n={len(h_spike)}, μ={mean_s:.2f} m)')
            plotted_any = True

        # ---- 전체 Normal/Spike 평균 세로선 (모든 subplot 공통) ----
        # 첫 번째 subplot에만 legend entry 추가
        is_first_axis = (cid == class_ids[0])

        if global_mean_normal is not None:
            gln = ax.axvline(global_mean_normal,
                             color='green', linestyle=':', linewidth=1.5, alpha=0.6)
            if is_first_axis:
                handles.append(gln)
                labels.append(f'Global Normal μ={global_mean_normal:.2f} m')

        if global_mean_spike is not None:
            gls = ax.axvline(global_mean_spike,
                             color='red', linestyle=':', linewidth=1.5, alpha=0.6)
            if is_first_axis:
                handles.append(gls)
                labels.append(f'Global Spike μ={global_mean_spike:.2f} m')

        if not plotted_any:
            ax.text(0.5, 0.5, 'No data', transform=ax.transAxes,
                    ha='center', va='center', fontsize=11)
        else:
            ax.legend(handles, labels, loc='lower right', fontsize=9)

        ax.set_ylabel('CDF', fontsize=11)
        ax.set_title(f'{label}', fontsize=12, fontweight='bold')
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel('Height (m)', fontsize=12)
    axes[-1].set_xlim(xmin, xmax)

    # 전체 제목 (Normal/Spike 정보 포함)
    fig.suptitle(
        f'ISA={fctimp_str} | Height CDF by Classification (Selected Pixels Only)\n'
        f'Normal: {normal_info["time_str"]} | Heading {normal_info["heading"]:.0f}°-{normal_info["tile"]} | Error={normal_info["error"]:.2f} m\n'
        f'Spike:  {spike_info["time_str"]} | Heading {spike_info["heading"]:.0f}°-{spike_info["tile"]} | Error={spike_info["error"]:.2f} m',
        fontsize=12, fontweight='bold'
    )

    plt.tight_layout(rect=[0, 0, 1, 0.88])

    out_path = f'{output_dir}/COMPARE_{time_str}_HeightCDFByClass.png'
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()

    print(f"  Saved: {out_path}")



def process_fctimp_height_analysis(fctimp_str, spike_threshold=1.0, output_dir=None):
    """Main function to process height by classification analysis"""
    
    if output_dir is None:
        output_dir = f'isa_{fctimp_str}_v3'
    Path(output_dir).mkdir(exist_ok=True)
    
    # Load reach polygon
    selected_12 = gpd.read_file('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/selected_12_vD.shp')
    fctimp_val = float(fctimp_str)
    sample = selected_12[np.isclose(selected_12["fctimp_mea"], fctimp_val, atol=0.01)].copy()
    if sample.empty:
        print(f"No reach for {fctimp_str}")
        return
    
    # Load time series data
    df = pd.read_parquet(f'/work/pi_kandread_umass_edu/swot-urban/1_data_processing/fctimp_mean_{fctimp_str}_sample.parquet')
    d = df.copy()
    d['time_dt'] = pd.to_datetime(d['time_str'])
    d = d.sort_values('time_dt').reset_index(drop=True)
    d['error'] = d['wse_relative_m'] - d['wse_usgs_relative_m']  # + = overestimate, - = underestimate
    d['is_spike'] = d['error'] >= spike_threshold
    
    # Load PIXC files
    pixc_dir = Path(f'/project/pi_kandread_umass_edu/swot-urban/0_data_collection/SWOT_PIXC/fctimp_{fctimp_str}')
    pixcvec_dir = Path(f'/project/pi_kandread_umass_edu/swot-urban/0_data_collection/SWOT_PIXCVec/fctimp_{fctimp_str}')
    pixc_files = filter_best_pixc_files(list(pixc_dir.glob('SWOT_L2_HR_PIXC_*.nc')))
    pixcvec_files = filter_best_pixc_files(list(pixcvec_dir.glob('SWOT_L2_HR_PIXCVec_*.nc')))
    
    polygon = sample.to_crs('EPSG:4326')
    
    print(f"\n{'='*70}")
    print(f"Processing ISA={fctimp_str}")
    print(f"Spikes: {d['is_spike'].sum()}, Normals: {(~d['is_spike']).sum()}")
    print(f"{'='*70}")
    
    for idx, spike_row in d[d['is_spike']].iterrows():
        print(f"\nSpike: {spike_row['time_str']} (error={spike_row['error']:.2f}m)")
        
        # Find closest normal
        normals = d[~d['is_spike']].copy()
        if len(normals) == 0:
            continue
        normals['time_diff'] = (normals['time_dt'] - spike_row['time_dt']).abs()
        closest_normal = normals.loc[normals['time_diff'].idxmin()]
        print(f"Normal: {closest_normal['time_str']} (error={closest_normal['error']:.2f}m)")
        
        # Get Normal data (selected pixels only)
        normal_data = None
        normal_info = None
        for group_key, nf_list in match_and_group_pixc(closest_normal['time_str'], pixc_files, closest_normal['crid']).items():
            vec_list = [v for v in [find_matching_pixcvec(nf, pixcvec_files) for nf in nf_list] if v]
            if not vec_list:
                continue
            data = extract_and_merge_selected_pixels(nf_list, vec_list, polygon)
            if data:
                # Get heading from first file
                ds_tvp = xr.open_dataset(nf_list[0], group='tvp')
                heading = np.nanmedian(ds_tvp['velocity_heading'].values)
                ds_tvp.close()
                
                normal_data = data
                normal_info = {
                    'time_str': closest_normal['time_dt'].strftime('%Y%m%dT%H%M%S'),
                    'time_dt': closest_normal['time_dt'],
                    'crid': closest_normal['crid'],
                    'error': closest_normal['error'],
                    'tile': get_tile_from_filename(nf_list[0].name),
                    'heading': heading,
                }
                break
        
        # Get Spike data (selected pixels only)
        spike_data = None
        spike_info = None
        for group_key, sf_list in match_and_group_pixc(spike_row['time_str'], pixc_files, spike_row['crid']).items():
            vec_list = [v for v in [find_matching_pixcvec(sf, pixcvec_files) for sf in sf_list] if v]
            if not vec_list:
                continue
            data = extract_and_merge_selected_pixels(sf_list, vec_list, polygon)
            if data:
                # Get heading from first file
                ds_tvp = xr.open_dataset(sf_list[0], group='tvp')
                heading = np.nanmedian(ds_tvp['velocity_heading'].values)
                ds_tvp.close()
                
                spike_data = data
                spike_info = {
                    'time_str': spike_row['time_dt'].strftime('%Y%m%dT%H%M%S'),
                    'time_dt': spike_row['time_dt'],
                    'crid': spike_row['crid'],
                    'error': spike_row['error'],
                    'tile': get_tile_from_filename(sf_list[0].name),
                    'heading': heading,
                }
                break
        
        # Generate CDF plot instead of boxplot
        if normal_data and spike_data:
            plot_height_cdf_by_classification(
                normal_data, spike_data,
                normal_info, spike_info,
                output_dir,
                spike_info['time_str'],
                fctimp_str
            )
        else:
            print(f"  Skipped: Missing data (Normal={normal_data is not None}, Spike={spike_data is not None})")

    print(f"\n✓ Done: {output_dir}")


# Run for ISA=0.01
if __name__ == "__main__":
    process_fctimp_height_analysis('0.01', spike_threshold=0.1832, output_dir='isa_0.01_v3')

In [ ]:
for fc in ['0.01', '0.06', '0.12', '0.17', '0.22', '0.32', '0.28', '0.40', '0.45', '0.50', '0.55', '0.61']:
    process_fctimp_height_analysis(fc, spike_threshold=0.1832, output_dir=f'isa_{fc}_v3')

#### Comparison only for similar spots

In [ ]:
def filter_best_pixc_files(files):
    file_dict = {}
    for f in files:
        match = re.match(r'(.+)_(P[IG]D\d)_(\d+)\.nc', f.name)
        if not match:
            continue
        base, processor, counter = match.group(1), match.group(2), int(match.group(3))
        if base not in file_dict:
            file_dict[base] = (f, processor, counter)
        else:
            _, old_proc, old_cnt = file_dict[base]
            if (processor == 'PGD0' and old_proc == 'PID0') or (processor == old_proc and counter > old_cnt):
                file_dict[base] = (f, processor, counter)
    return [f for f, _, _ in file_dict.values()]

def get_tile_from_filename(filename):
    parts = filename.split('_')
    if len(parts) >= 7:
        return parts[6]
    return 'UNK'

def match_and_group_pixc(time_str, files, crid, window_minutes=5):
    dt = pd.to_datetime(time_str)
    if dt.tzinfo:
        dt = dt.tz_localize(None)
    start, end = dt - timedelta(minutes=window_minutes), dt + timedelta(minutes=window_minutes)
    groups = {}
    for f in files:
        parts = f.name.split('_')
        if len(parts) >= 10:
            try:
                fs, fe = pd.to_datetime(parts[7], format='%Y%m%dT%H%M%S'), pd.to_datetime(parts[8], format='%Y%m%dT%H%M%S')
                if not (fe < start or fs > end) and parts[9] == crid:
                    key = f"{parts[7]}_{parts[8]}"
                    groups.setdefault(key, []).append(f)
            except:
                continue
    return groups

def find_matching_pixcvec(pixc_file, pixcvec_files):
    pixc_name = pixc_file.name.replace('PIXC_', 'PIXCVec_')
    for pvf in pixcvec_files:
        if pvf.name == pixc_name:
            return pvf
    return None

def extract_selected_pixels(pixc_file, pixcvec_file, polygon_gdf):
    """Extract ONLY selected pixels (via PIXCVec) within polygon"""
    ds = xr.open_dataset(pixc_file, group='pixel_cloud')
    lon, lat = ds['longitude'].values, ds['latitude'].values
    bounds = polygon_gdf.total_bounds
    
    bbox_mask = ((lon >= bounds[0]) & (lon <= bounds[2]) & (lat >= bounds[1]) & (lat <= bounds[3]))
    if bbox_mask.sum() == 0:
        ds.close()
        return None
    
    classification = ds['classification'].values[bbox_mask]
    water_mask = classification > 2
    if water_mask.sum() == 0:
        ds.close()
        return None
    
    lon_filt, lat_filt = lon[bbox_mask][water_mask], lat[bbox_mask][water_mask]
    poly = polygon_gdf.geometry.iloc[0]
    inside_mask = np.array([poly.contains(Point(x, y)) for x, y in zip(lon_filt, lat_filt)])
    if inside_mask.sum() == 0:
        ds.close()
        return None
    
    final_idx = np.where(bbox_mask)[0][water_mask][inside_mask]
    
    # Get PIXCVec selection
    ds_vec = xr.open_dataset(pixcvec_file)
    reach_id = ds_vec['reach_id'].values[final_idx]
    node_id = ds_vec['node_id'].values[final_idx]
    obs_id = ds_vec['obs_id'].values[final_idx]
    lake_id = ds_vec['lake_id'].values[final_idx]
    
    def is_valid(arr):
        return np.array([len(x) > 0 and x != b'' for x in arr]) if arr.dtype.kind == 'S' else arr > 0
    
    selected_mask = is_valid(reach_id) & is_valid(node_id) & ~is_valid(obs_id) & ~is_valid(lake_id)
    
    # Apply selection mask - ONLY return selected pixels
    selected_idx = final_idx[selected_mask]
    
    if len(selected_idx) == 0:
        ds.close()
        ds_vec.close()
        return None
    
    result = {
        'lon': lon[selected_idx],
        'lat': lat[selected_idx],
        'height': ds['height'].values[selected_idx],
        'classification': ds['classification'].values[selected_idx],
    }
    
    ds.close()
    ds_vec.close()
    return result

def extract_and_merge_selected_pixels(pixc_files, pixcvec_files, polygon_gdf):
    """Extract and merge selected pixels from multiple tiles"""
    all_data = []
    for pf, pvf in zip(pixc_files, pixcvec_files):
        data = extract_selected_pixels(pf, pvf, polygon_gdf)
        if data:
            all_data.append(data)
    
    if not all_data:
        return None
    if len(all_data) == 1:
        return all_data[0]
    
    merged = {k: np.concatenate([d[k] for d in all_data]) for k in all_data[0].keys()}
    return merged


def match_pixels_nearest_neighbor(normal_data, spike_data, threshold_deg=0.0001):
    """
    Match pixels between Normal and Spike using Nearest Neighbor (KD-Tree)
    
    Parameters:
        normal_data: dict with 'lon', 'lat' arrays
        spike_data: dict with 'lon', 'lat' arrays
        threshold_deg: max distance in degrees (~11m at 0.0001°)
    
    Returns:
        normal_idx: indices of matched Normal pixels
        spike_idx: indices of matched Spike pixels
        distances: distances between matched pairs
    """
    # Build KD-Tree from Normal pixels
    normal_coords = np.column_stack([normal_data['lon'], normal_data['lat']])
    spike_coords = np.column_stack([spike_data['lon'], spike_data['lat']])
    
    normal_tree = cKDTree(normal_coords)
    
    # Find nearest Normal pixel for each Spike pixel
    distances, normal_indices = normal_tree.query(spike_coords, k=1)
    
    # Filter by threshold
    matched_mask = distances <= threshold_deg
    
    # Get matched indices
    spike_idx = np.where(matched_mask)[0]
    normal_idx = normal_indices[matched_mask]
    matched_distances = distances[matched_mask]
    
    return normal_idx, spike_idx, matched_distances


def plot_matched_pixel_analysis(normal_data, spike_data, normal_info, spike_info, output_dir, time_str, fctimp_str):
    """
    Analyze matched pixels between Normal and Spike
    1. Find same-location pixels using Nearest Neighbor
    2. Calculate height difference from open_water-to-open_water pixels
    3. Compare classification changes with normalized heights
    """
    
    # Step 1: Match pixels using Nearest Neighbor
    # threshold_deg=0.0001 ≈ 11m at equator
    threshold_deg = 0.0001
    normal_idx, spike_idx, distances = match_pixels_nearest_neighbor(
        normal_data, spike_data, threshold_deg=threshold_deg
    )
    
    if len(normal_idx) == 0:
        print(f"  No matched pixels found within threshold ({threshold_deg}° ≈ {threshold_deg * 111000:.1f}m)!")
        return
    
    mean_dist_m = np.mean(distances) * 111000  # approximate meters
    max_dist_m = np.max(distances) * 111000
    
    print(f"  Matched {len(normal_idx)} pixels (threshold: {threshold_deg}° ≈ {threshold_deg * 111000:.1f}m)")
    print(f"  Mean distance: {mean_dist_m:.2f}m, Max distance: {max_dist_m:.2f}m")
    
    # Extract matched data
    matched_normal = {
        'lon': normal_data['lon'][normal_idx],
        'lat': normal_data['lat'][normal_idx],
        'height': normal_data['height'][normal_idx],
        'classification': normal_data['classification'][normal_idx],
    }
    matched_spike = {
        'lon': spike_data['lon'][spike_idx],
        'lat': spike_data['lat'][spike_idx],
        'height': spike_data['height'][spike_idx],
        'classification': spike_data['classification'][spike_idx],
    }
    
    # Step 2: Find open_water (class 4) in BOTH Normal and Spike
    # These are "stable" pixels for calculating baseline height difference
    open_water_both = (matched_normal['classification'] == 4) & (matched_spike['classification'] == 4)
    n_open_water_both = open_water_both.sum()
    
    if n_open_water_both == 0:
        print(f"  No open_water-to-open_water pixels found!")
        return
    
    # Calculate mean height difference (Spike - Normal) for open_water pixels
    height_diff_open_water = matched_spike['height'][open_water_both] - matched_normal['height'][open_water_both]
    mean_height_diff = np.nanmean(height_diff_open_water)
    
    print(f"  Open water (both): {n_open_water_both} pixels")
    print(f"  Mean height difference (Spike - Normal): {mean_height_diff:.3f} m")
    
    # Step 3: Find pixels that are open_water in Normal but changed in Spike
    # Normal: open_water (4) -> Spike: low_coh_water_near_land (6) or open_low_coh_water (7)
    normal_open_spike_lcwnl = (matched_normal['classification'] == 4) & (matched_spike['classification'] == 6)
    normal_open_spike_olcw = (matched_normal['classification'] == 4) & (matched_spike['classification'] == 7)
    
    n_lcwnl = normal_open_spike_lcwnl.sum()
    n_olcw = normal_open_spike_olcw.sum()
    
    print(f"  Normal(open_water) -> Spike(low_coh_water_near_land): {n_lcwnl} pixels")
    print(f"  Normal(open_water) -> Spike(open_low_coh_water): {n_olcw} pixels")
    
    # Get heights for each group
    # Normalize spike heights by subtracting mean_height_diff
    heights_normal_open = matched_normal['height'][open_water_both]
    heights_spike_open = matched_spike['height'][open_water_both] - mean_height_diff  # normalized
    
    heights_normal_to_lcwnl = matched_normal['height'][normal_open_spike_lcwnl]
    heights_spike_lcwnl = matched_spike['height'][normal_open_spike_lcwnl] - mean_height_diff  # normalized
    
    heights_normal_to_olcw = matched_normal['height'][normal_open_spike_olcw]
    heights_spike_olcw = matched_spike['height'][normal_open_spike_olcw] - mean_height_diff  # normalized
    
    # =========================================================================
    # Additional classification groups: water_near_land (class 3)
    # =========================================================================
    # Both water_near_land (class 3)
    water_near_land_both = (matched_normal['classification'] == 3) & (matched_spike['classification'] == 3)
    n_wnl_both = water_near_land_both.sum()
    
    heights_normal_wnl = matched_normal['height'][water_near_land_both]
    heights_spike_wnl_raw = matched_spike['height'][water_near_land_both]  # RAW (no correction)
    
    print(f"  Water near land (both): {n_wnl_both} pixels")
    
    # RAW spike heights for open_water (before correction)
    heights_spike_open_raw = matched_spike['height'][open_water_both]  # RAW
    
    # Calculate overall means (all matched pixels)
    all_normal_heights = matched_normal['height']
    all_spike_heights_raw = matched_spike['height']
    overall_mean_normal = np.nanmean(all_normal_heights)
    overall_mean_spike_raw = np.nanmean(all_spike_heights_raw)
    
    # =========================================================================
    # CDF PLOT: 4 rows with shared x-axis
    # Row 0: both water_near_land (RAW - no correction)
    # Row 1: both open_water (RAW - shows where correction comes from)
    # Row 2: changed to low_coh_water_near_land (CORRECTED)
    # Row 3: changed to open_low_coh_water (CORRECTED)
    # =========================================================================
    
    def plot_cdf(ax, data, color, label, linestyle='-', linewidth=2):
        """Plot CDF on given axis"""
        if len(data) == 0:
            return None, None
        sorted_data = np.sort(data[np.isfinite(data)])
        cdf = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
        ax.plot(sorted_data, cdf, color=color, linestyle=linestyle, linewidth=linewidth, label=label)
        return np.nanmean(data), len(data)
    
    # Collect all heights to determine shared x-axis range
    all_heights = []
    if n_wnl_both > 0:
        all_heights.extend([heights_normal_wnl, heights_spike_wnl_raw])
    if n_open_water_both > 0:
        all_heights.extend([heights_normal_open, heights_spike_open_raw])
    if n_lcwnl > 0:
        all_heights.extend([heights_normal_to_lcwnl, heights_spike_lcwnl])
    if n_olcw > 0:
        all_heights.extend([heights_normal_to_olcw, heights_spike_olcw])
    
    if len(all_heights) == 0:
        print(f"  No data for CDF plot!")
        return
    
    all_heights_flat = np.concatenate([h for h in all_heights if len(h) > 0])
    x_min = np.nanpercentile(all_heights_flat, 1) - 0.5
    x_max = np.nanpercentile(all_heights_flat, 99) + 0.5
    
    # Create figure with 4 rows
    fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
    
    row_configs = [
        {
            'title': 'Both water_near_land (class 3) - RAW',
            'normal_data': heights_normal_wnl if n_wnl_both > 0 else np.array([]),
            'spike_data': heights_spike_wnl_raw if n_wnl_both > 0 else np.array([]),
            'n_pixels': n_wnl_both,
            'is_corrected': False,
            'spike_label': 'Spike RAW',
        },
        {
            'title': f'Both open_water (class 4) - RAW → Height correction calculated here: Δ={mean_height_diff:.3f}m',
            'normal_data': heights_normal_open if n_open_water_both > 0 else np.array([]),
            'spike_data': heights_spike_open_raw if n_open_water_both > 0 else np.array([]),
            'n_pixels': n_open_water_both,
            'is_corrected': False,
            'spike_label': 'Spike RAW',
        },
        {
            'title': f'Changed: open_water → low_coh_water_near_land - CORRECTED (Spike - {mean_height_diff:.3f}m)',
            'normal_data': heights_normal_to_lcwnl if n_lcwnl > 0 else np.array([]),
            'spike_data': heights_spike_lcwnl if n_lcwnl > 0 else np.array([]),
            'n_pixels': n_lcwnl,
            'is_corrected': True,
            'spike_label': 'Spike CORRECTED',
        },
        {
            'title': f'Changed: open_water → open_low_coh_water - CORRECTED (Spike - {mean_height_diff:.3f}m)',
            'normal_data': heights_normal_to_olcw if n_olcw > 0 else np.array([]),
            'spike_data': heights_spike_olcw if n_olcw > 0 else np.array([]),
            'n_pixels': n_olcw,
            'is_corrected': True,
            'spike_label': 'Spike CORRECTED',
        },
    ]
    
    for i, (ax, config) in enumerate(zip(axes, row_configs)):
        normal_data = config['normal_data']
        spike_data = config['spike_data']
        n_pixels = config['n_pixels']
        is_corrected = config['is_corrected']
        spike_label = config['spike_label']
        
        if n_pixels > 0 and len(normal_data) > 0 and len(spike_data) > 0:
            # Plot CDFs
            mean_normal, n_normal = plot_cdf(ax, normal_data, 'green', f'Normal (n={len(normal_data)})')
            mean_spike, n_spike = plot_cdf(ax, spike_data, 'red', f'{spike_label} (n={len(spike_data)})')
            
            # Add mean lines as vertical dashed lines
            if mean_normal is not None:
                ax.axvline(x=mean_normal, color='green', linestyle='--', linewidth=1.5, alpha=0.7)
            if mean_spike is not None:
                ax.axvline(x=mean_spike, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
            
            # Add overall mean lines (thinner, different style)
            ax.axvline(x=overall_mean_normal, color='darkgreen', linestyle=':', linewidth=1, alpha=0.5)
            ax.axvline(x=overall_mean_spike_raw, color='darkred', linestyle=':', linewidth=1, alpha=0.5)
            
            # Stats text
            diff = mean_spike - mean_normal if (mean_normal is not None and mean_spike is not None) else 0
            stats_text = f'n={n_pixels} | Normal μ={mean_normal:.2f}m | Spike μ={mean_spike:.2f}m | Δ={diff:.3f}m'
            ax.text(0.98, 0.95, stats_text, transform=ax.transAxes, 
                    ha='right', va='top', fontsize=9, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='gray', alpha=0.9))
            
            # Highlight row 1 (open_water) as the source of correction
            if i == 1:
                ax.set_facecolor('#ffffee')  # Light yellow background
            
            ax.legend(loc='lower right', fontsize=9)
        else:
            ax.text(0.5, 0.5, 'No data', transform=ax.transAxes, 
                    ha='center', va='center', fontsize=12, color='gray')
        
        ax.set_ylabel('CDF', fontsize=10)
        ax.set_title(config['title'], fontsize=10, fontweight='bold', loc='left',
                     color='darkorange' if i == 1 else 'black')
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0, 1.05)
        ax.set_xlim(x_min, x_max)
    
    # X-axis label only on bottom
    axes[-1].set_xlabel('Height (m)', fontsize=11)
    
    # Main title with overall stats
    fig.suptitle(f'ISA={fctimp_str} | CDF Comparison: Normal vs Spike\n'
                 f'Matched pixels (NN, {threshold_deg}°≈{threshold_deg*111000:.0f}m): {len(normal_idx)} | '
                 f'Overall Mean: Normal={overall_mean_normal:.2f}m, Spike RAW={overall_mean_spike_raw:.2f}m',
                 fontsize=12, fontweight='bold')
    
    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.savefig(f'{output_dir}/MATCHED_{time_str}_CDF.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"  Saved: MATCHED_{time_str}_CDF.png")


def process_fctimp_matched_analysis(fctimp_str, spike_threshold=1.0, output_dir=None):
    """Main function to process matched pixel analysis"""
    
    if output_dir is None:
        output_dir = f'isa_{fctimp_str}_v3'
    Path(output_dir).mkdir(exist_ok=True)
    
    # Load reach polygon
    selected_12 = gpd.read_file('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/selected_12_vD.shp')
    fctimp_val = float(fctimp_str)
    sample = selected_12[np.isclose(selected_12["fctimp_mean"], fctimp_val, atol=0.01)].copy()
    if sample.empty:
        print(f"No reach for {fctimp_str}")
        return
    
    # Load time series data
    df = pd.read_parquet(f'/work/pi_kandread_umass_edu/swot-urban/1_data_processing/fctimp_meann_{fctimp_str}_sample.parquet')
    d = df.copy()
    d['time_dt'] = pd.to_datetime(d['time_str'])
    d = d.sort_values('time_dt').reset_index(drop=True)
    d['error'] = d['wse_relative_m'] - d['wse_usgs_relative_m']  # + = overestimate, - = underestimate
    d['is_spike'] = d['error'] >= spike_threshold
    
    # Load PIXC files
    pixc_dir = Path(f'/project/pi_kandread_umass_edu/swot-urban/0_data_collection/SWOT_PIXC/fctimp_{fctimp_str}')
    pixcvec_dir = Path(f'/project/pi_kandread_umass_edu/swot-urban/0_data_collection/SWOT_PIXCVec/fctimp_{fctimp_str}')
    pixc_files = filter_best_pixc_files(list(pixc_dir.glob('SWOT_L2_HR_PIXC_*.nc')))
    pixcvec_files = filter_best_pixc_files(list(pixcvec_dir.glob('SWOT_L2_HR_PIXCVec_*.nc')))
    
    polygon = sample.to_crs('EPSG:4326')
    
    print(f"\n{'='*70}")
    print(f"MATCHED PIXEL ANALYSIS (Nearest Neighbor): ISA={fctimp_str}")
    print(f"Spikes: {d['is_spike'].sum()}, Normals: {(~d['is_spike']).sum()}")
    print(f"{'='*70}")
    
    for idx, spike_row in d[d['is_spike']].iterrows():
        print(f"\n{'-'*50}")
        print(f"Spike: {spike_row['time_str']} (error={spike_row['error']:.2f}m)")
        
        # Find closest normal
        normals = d[~d['is_spike']].copy()
        if len(normals) == 0:
            continue
        normals['time_diff'] = (normals['time_dt'] - spike_row['time_dt']).abs()
        closest_normal = normals.loc[normals['time_diff'].idxmin()]
        print(f"Normal: {closest_normal['time_str']} (error={closest_normal['error']:.2f}m)")
        
        # Get Normal data (selected pixels only)
        normal_data = None
        normal_info = None
        for group_key, nf_list in match_and_group_pixc(closest_normal['time_str'], pixc_files, closest_normal['crid']).items():
            vec_list = [v for v in [find_matching_pixcvec(nf, pixcvec_files) for nf in nf_list] if v]
            if not vec_list:
                continue
            data = extract_and_merge_selected_pixels(nf_list, vec_list, polygon)
            if data:
                ds_tvp = xr.open_dataset(nf_list[0], group='tvp')
                heading = np.nanmedian(ds_tvp['velocity_heading'].values)
                ds_tvp.close()
                
                normal_data = data
                normal_info = {
                    'time_str': closest_normal['time_dt'].strftime('%Y%m%dT%H%M%S'),
                    'time_dt': closest_normal['time_dt'],
                    'error': closest_normal['error'],
                    'tile': get_tile_from_filename(nf_list[0].name),
                    'heading': heading,
                }
                break
        
        # Get Spike data (selected pixels only)
        spike_data = None
        spike_info = None
        for group_key, sf_list in match_and_group_pixc(spike_row['time_str'], pixc_files, spike_row['crid']).items():
            vec_list = [v for v in [find_matching_pixcvec(sf, pixcvec_files) for sf in sf_list] if v]
            if not vec_list:
                continue
            data = extract_and_merge_selected_pixels(sf_list, vec_list, polygon)
            if data:
                ds_tvp = xr.open_dataset(sf_list[0], group='tvp')
                heading = np.nanmedian(ds_tvp['velocity_heading'].values)
                ds_tvp.close()
                
                spike_data = data
                spike_info = {
                    'time_str': spike_row['time_dt'].strftime('%Y%m%dT%H%M%S'),
                    'time_dt': spike_row['time_dt'],
                    'error': spike_row['error'],
                    'tile': get_tile_from_filename(sf_list[0].name),
                    'heading': heading,
                }
                break
        
        # Generate matched pixel analysis
        if normal_data and spike_data:
            print(f"  Normal pixels: {len(normal_data['lon'])}")
            print(f"  Spike pixels: {len(spike_data['lon'])}")
            plot_matched_pixel_analysis(normal_data, spike_data, normal_info, spike_info, 
                                        output_dir, spike_info['time_str'], fctimp_str)
        else:
            print(f"  Skipped: Missing data")
    
    print(f"\n✓ Done: {output_dir}")


# Run for ISA=0.01
if __name__ == "__main__":
    process_fctimp_matched_analysis('0.01', spike_threshold=0.1832, output_dir='isa_0.01_v3')

In [ ]:
for fc in ['0.01', '0.06', '0.12', '0.17', '0.22', '0.32', '0.28', '0.40', '0.45', '0.50', '0.55', '0.61']:
    process_fctimp_matched_analysis(fc, spike_threshold=0.1832, output_dir=f'isa_{fc}_v3')

### PIXC SPIKE & NORMAL BY ISA (NLCD, Sig0, Classification, Height)

#### single

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter
import rasterio
from pathlib import Path
import xarray as xr
from shapely.geometry import Point
from datetime import datetime, timedelta
import math
import re

# Classification colors and labels (PIXC)
CLASSIFICATION_COLORS = [
    'maroon',        # 1: land
    'gold',          # 2: land_near_water
    'skyblue',       # 3: water_near_land
    'mediumblue',    # 4: open_water
    'dimgray',       # 5: dark_water
    'cyan',          # 6: low_coh_water_near_land
    'darkturquoise', # 7: open_low_coh_water
]

CLASSIFICATION_LABELS = [
    'Land',
    'Land (near water)',
    'Water (near land)',
    'Open Water',
    'Dark Water',
    'Low Coherence (near land)',
    'Low Coherence (open)'
]

# NLCD colors and labels
NLCD_COLORS = {
    11: '#476BA1', 12: '#D1DDF9', 21: '#DDC9C9', 22: '#D89382',
    23: '#ED0000', 24: '#AA0000', 31: '#B2ADA3', 41: '#68AA63',
    42: '#1C6330', 43: '#B5C98E', 52: '#CCBA7C', 71: '#E2E2C1',
    81: '#DDD83E', 82: '#AE7229', 90: '#BBD7ED', 95: '#71A4C1'
}

NLCD_LABELS = {
    11: 'Open Water', 12: 'Ice/Snow',
    21: 'Developed (Open)', 22: 'Developed (Low)',
    23: 'Developed (Med)', 24: 'Developed (High)',
    31: 'Barren', 41: 'Deciduous Forest',
    42: 'Evergreen Forest', 43: 'Mixed Forest',
    52: 'Shrub/Scrub', 71: 'Grassland',
    81: 'Pasture/Hay', 82: 'Cultivated',
    90: 'Woody Wetlands', 95: 'Herbaceous Wetlands'
}

POLYGON_COLOR = '#404040'


def extract_pixels_with_vec(pixc_file, pixcvec_file, polygon_gdf):
    ds = xr.open_dataset(pixc_file, group='pixel_cloud')
    lon, lat = ds['longitude'].values, ds['latitude'].values
    bounds = polygon_gdf.total_bounds
    
    # Get tvp group data (satellite position)
    ds_tvp = xr.open_dataset(pixc_file, group='tvp')
    heading = np.nanmedian(ds_tvp['velocity_heading'].values)
    
    # Satellite position - use median values
    sat_lon = np.nanmedian(ds_tvp['longitude'].values)
    sat_lat = np.nanmedian(ds_tvp['latitude'].values)
    sat_alt = np.nanmedian(ds_tvp['altitude'].values)
    ds_tvp.close()
    
    bbox_mask = ((lon >= bounds[0]) & (lon <= bounds[2]) & (lat >= bounds[1]) & (lat <= bounds[3]))
    if bbox_mask.sum() == 0:
        ds.close()
        return None
    
    classification = ds['classification'].values[bbox_mask]
    water_mask = classification > 2
    if water_mask.sum() == 0:
        ds.close()
        return None
    
    lon_filt, lat_filt = lon[bbox_mask][water_mask], lat[bbox_mask][water_mask]
    poly = polygon_gdf.geometry.iloc[0]
    inside_mask = np.array([poly.contains(Point(x, y)) for x, y in zip(lon_filt, lat_filt)])
    if inside_mask.sum() == 0:
        ds.close()
        return None
    
    final_idx = np.where(bbox_mask)[0][water_mask][inside_mask]
    
    # Get cross-track distance
    cross_track = ds['cross_track'].values[final_idx]
    median_cross_track = np.nanmedian(cross_track)
    
    ds_vec = xr.open_dataset(pixcvec_file)
    reach_id = ds_vec['reach_id'].values[final_idx]
    node_id = ds_vec['node_id'].values[final_idx]
    obs_id = ds_vec['obs_id'].values[final_idx]
    lake_id = ds_vec['lake_id'].values[final_idx]
    
    def is_valid(arr):
        return np.array([len(x) > 0 and x != b'' for x in arr]) if arr.dtype.kind == 'S' else arr > 0
    
    selected_mask = is_valid(reach_id) & is_valid(node_id) & ~is_valid(obs_id) & ~is_valid(lake_id)
    
    result = {
        'lon': lon[final_idx], 'lat': lat[final_idx],
        'lon_all': lon, 'lat_all': lat,
        'height': ds['height'].values[final_idx],
        'classification': ds['classification'].values[final_idx],
        'bright_land': ds['bright_land_flag'].values[final_idx],
        'layover': ds['layover_impact'].values[final_idx],
        'sig0': ds['sig0'].values[final_idx],
        'coh_power': ds['coherent_power'].values[final_idx],
        'azimuth': ds['azimuth_index'].values[final_idx],
        'range': ds['range_index'].values[final_idx],
        'cross_track': cross_track,
        'selected_mask': selected_mask,
        'heading': heading,
        'sat_info': {
            'lon': sat_lon,
            'lat': sat_lat,
            'altitude': sat_alt,
            'cross_track': median_cross_track
        }
    }
    ds.close()
    ds_vec.close()
    return result


def extract_and_merge_pixels(pixc_files, pixcvec_files, polygon_gdf):
    all_data = [extract_pixels_with_vec(pf, pvf, polygon_gdf) for pf, pvf in zip(pixc_files, pixcvec_files)]
    all_data = [d for d in all_data if d]
    if not all_data:
        return None
    if len(all_data) == 1:
        return all_data[0]
    merged = {k: np.concatenate([d[k] for d in all_data]) for k in all_data[0].keys() 
              if k not in ['heading', 'sat_info']}
    merged['heading'] = np.nanmean([d['heading'] for d in all_data])
    # Use first satellite info (they should be very similar)
    merged['sat_info'] = all_data[0]['sat_info']
    return merged
    

def find_matching_pixcvec(pixc_file, pixcvec_files):
    pixc_name = pixc_file.name.replace('PIXC_', 'PIXCVec_')
    for pvf in pixcvec_files:
        if pvf.name == pixc_name:
            return pvf
    return None


def filter_best_pixc_files(files):
    file_dict = {}
    for f in files:
        match = re.match(r'(.+)_(P[IG]D\d)_(\d+)\.nc', f.name)
        if not match:
            continue
        base, processor, counter = match.group(1), match.group(2), int(match.group(3))
        if base not in file_dict:
            file_dict[base] = (f, processor, counter)
        else:
            _, old_proc, old_cnt = file_dict[base]
            if (processor == 'PGD0' and old_proc == 'PID0') or (processor == old_proc and counter > old_cnt):
                file_dict[base] = (f, processor, counter)
    return [f for f, _, _ in file_dict.values()]


def get_tile_from_filename(filename):
    parts = filename.split('_')
    if len(parts) >= 7:
        return parts[6]
    return 'UNK'


def get_pass_from_filename(filename):
    parts = filename.split('_')
    if len(parts) >= 6:
        return parts[5]
    return 'UNK'


def lon_formatter(x, pos):
    if x < 0:
        return f'{abs(x):.2f}°W'
    elif x > 0:
        return f'{x:.2f}°E'
    else:
        return '0°'


def lat_formatter(y, pos):
    if y > 0:
        return f'{y:.2f}°N'
    elif y < 0:
        return f'{abs(y):.2f}°S'
    else:
        return '0°'


def add_satellite_and_path(ax, range_data, azimuth_data, heading, sat_info, pass_num, tile):
    """
    Add satellite icon, orbit path line, heading arrow in range-azimuth space
    """
    xtrk = sat_info['cross_track']
    
    # Range-azimuth bounds
    range_min, range_max = np.nanmin(range_data), np.nanmax(range_data)
    az_min, az_max = np.nanmin(azimuth_data), np.nanmax(azimuth_data)
    
    range_span = range_max - range_min
    az_span = az_max - az_min
    
    # Satellite is at the near-range side (lower range values)
    sat_range = range_min - 0.15 * range_span
    sat_az = (az_min + az_max) / 2
    
    # Orbit path line (vertical in range-azimuth space = along-track)
    ax.axvline(x=sat_range, color='blue', linestyle='--', linewidth=1.5, alpha=0.7, clip_on=False)
    
    # Satellite marker
    ax.plot(sat_range, sat_az, 's', color='blue', markersize=15,
            markeredgecolor='white', markeredgewidth=1.5, zorder=20, clip_on=False)
    ax.text(sat_range, sat_az, 'SAT', fontsize=6, ha='center', va='center',
            color='white', fontweight='bold', zorder=21, clip_on=False)
    
    # Heading arrow (along azimuth direction)
    arrow_len = 0.1 * az_span
    ax.annotate('', xy=(sat_range, sat_az + arrow_len),
                xytext=(sat_range, sat_az),
                arrowprops=dict(arrowstyle='->', color='blue', lw=2),
                annotation_clip=False, zorder=19)
    
    # Heading label
    ax.text(sat_range, sat_az + arrow_len * 1.5, f'{heading:.0f}°',
            fontsize=10, fontweight='bold', color='blue',
            ha='center', va='bottom', clip_on=False)
    
    # Pass/Tile info
    info_text = f'Pass {pass_num}\nTile {tile}\nXtrk: {xtrk/1000:.1f} km'
    ax.text(sat_range, sat_az - 0.2 * az_span, info_text,
            fontsize=8, ha='center', va='top', color='blue', clip_on=False,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='blue', alpha=0.9))

def process_fctimp_comparison(fctimp_str, spike_threshold=0.1832, output_dir='figures', year=2025):
    """
    Spike (error >= threshold) vs Normal (error < threshold, 시간적으로 가장 가까운) 비교
    """
    Path(output_dir).mkdir(exist_ok=True)
    
    # 경로 설정
    selected_12_path = '/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/selected_12_vD.shp'
    parquet_path = f'/work/pi_kandread_umass_edu/swot-urban/1_data_processing/fctimp_mean_{fctimp_str}_sample.parquet'
    pixc_dir = Path(f'/project/pi_kandread_umass_edu/swot-urban/0_data_collection/SWOT_PIXC/fctimp_{fctimp_str}')
    pixcvec_dir = Path(f'/project/pi_kandread_umass_edu/swot-urban/0_data_collection/SWOT_PIXCVec/fctimp_{fctimp_str}')
    nlcd_path = '/work/pi_kandread_umass_edu/swot-urban/0_data_collection/1_nlcd/Annual_NLCD_LndCov_2024_CU_C1V1_EPSG4326.tif'
    
    # Load data
    selected_12 = gpd.read_file(selected_12_path)
    fctimp_val = float(fctimp_str)
    sample = selected_12[np.isclose(selected_12["fctimp_mea"], fctimp_val, atol=0.01)].copy()
    
    if sample.empty:
        print(f"No reach for {fctimp_str}")
        return
    
    df = pd.read_parquet(parquet_path)
    d = df.copy()
    d['time_dt'] = pd.to_datetime(d['time_str']).dt.tz_localize(None)
    d = d.sort_values('time_dt').reset_index(drop=True)
    
    # 2025년만 필터링
    d = d[d['time_dt'].dt.year == year].reset_index(drop=True)
    
    d['error'] = d['error_relative_m']
    d['is_spike'] = d['error'].abs() >= spike_threshold
    
    n_spikes = d['is_spike'].sum()
    n_normals = (~d['is_spike']).sum()
    
    print(f"ISA = {fctimp_str} ({year}년): {len(d)} observations")
    print(f"  Spikes (|error| >= {spike_threshold}m): {n_spikes}")
    print(f"  Normals (|error| < {spike_threshold}m): {n_normals}")
    
    if n_spikes == 0:
        print("  → No spikes found, nothing to compare")
        return
    
    if n_normals == 0:
        print("  → No normals found, nothing to compare")
        return
    
    # Load PIXC files
    pixc_files = filter_best_pixc_files(list(pixc_dir.glob('SWOT_L2_HR_PIXC_*.nc')))
    pixcvec_files = filter_best_pixc_files(list(pixcvec_dir.glob('SWOT_L2_HR_PIXCVec_*.nc')))
    
    print(f"  PIXC files: {len(pixc_files)}")
    print(f"  PIXCVec files: {len(pixcvec_files)}")
    
    polygon = sample.to_crs('EPSG:4326')
    bounds = polygon.total_bounds
    
    # Process each spike
    for idx, spike_row in d[d['is_spike']].iterrows():
        print(f"\n{'='*70}")
        print(f"Spike: {spike_row['time_dt'].strftime('%Y%m%dT%H%M%S')} (error={spike_row['error']:.2f}m)")
        
        # Find closest normal (시간적으로 가장 가까운)
        normals = d[~d['is_spike']].copy()
        normals['time_diff'] = (normals['time_dt'] - spike_row['time_dt']).abs()
        closest_normal = normals.loc[normals['time_diff'].idxmin()]
        
        print(f"Normal: {closest_normal['time_dt'].strftime('%Y%m%dT%H%M%S')} (error={closest_normal['error']:.2f}m, Δ={closest_normal['time_diff']})")
        
        # Get Normal data
        normal_data = None
        normal_info = None
        for group_key, nf_list in match_and_group_pixc(closest_normal['time_dt'], pixc_files, closest_normal['crid']).items():
            vec_list = [v for v in [find_matching_pixcvec(nf, pixcvec_files) for nf in nf_list] if v]
            if not vec_list:
                continue
            data = extract_and_merge_pixels(nf_list, vec_list, polygon)
            if data:
                normal_data = data
                normal_info = {
                    'time_str': closest_normal['time_dt'].strftime('%Y%m%dT%H%M%S'),
                    'time_dt': closest_normal['time_dt'],
                    'crid': closest_normal['crid'],
                    'error': closest_normal['error'],
                    'tile': get_tile_from_filename(nf_list[0].name),
                    'heading': data['heading'],
                    'group_key': group_key,
                    'sat_info': data['sat_info']
                }
                break
        
        # Get Spike data
        spike_data = None
        spike_info = None
        for group_key, sf_list in match_and_group_pixc(spike_row['time_dt'], pixc_files, spike_row['crid']).items():
            vec_list = [v for v in [find_matching_pixcvec(sf, pixcvec_files) for sf in sf_list] if v]
            if not vec_list:
                continue
            data = extract_and_merge_pixels(sf_list, vec_list, polygon)
            if data:
                spike_data = data
                spike_info = {
                    'time_str': spike_row['time_dt'].strftime('%Y%m%dT%H%M%S'),
                    'time_dt': spike_row['time_dt'],
                    'crid': spike_row['crid'],
                    'error': spike_row['error'],
                    'tile': get_tile_from_filename(sf_list[0].name),
                    'heading': data['heading'],
                    'group_key': group_key,
                    'sat_info': data['sat_info']
                }
                break
        
        # Plot comparison if both exist
        if normal_data and spike_data:
            save_comparison_plot(normal_data, spike_data, normal_info, spike_info, 
                                polygon, bounds, output_dir, fctimp_str, nlcd_path)
        else:
            if not normal_data:
                print("  → Could not extract normal data")
            if not spike_data:
                print("  → Could not extract spike data")
    
    print(f"\n✓ Done: {output_dir}")


def match_and_group_pixc(time_dt, files, crid, window_minutes=5):
    """Match PIXC files by time and CRID"""
    if hasattr(time_dt, 'tz_localize'):
        pass  # already naive
    start, end = time_dt - timedelta(minutes=window_minutes), time_dt + timedelta(minutes=window_minutes)
    groups = {}
    for f in files:
        parts = f.name.split('_')
        if len(parts) >= 10:
            try:
                fs = pd.to_datetime(parts[7], format='%Y%m%dT%H%M%S')
                fe = pd.to_datetime(parts[8], format='%Y%m%dT%H%M%S')
                if not (fe < start or fs > end) and parts[9] == crid:
                    key = f"{parts[7]}_{parts[8]}"
                    groups.setdefault(key, []).append(f)
            except:
                continue
    return groups


def save_comparison_plot(normal_data, spike_data, normal_info, spike_info, polygon, bounds, output_dir, fctimp_str, nlcd_path):
    """Save spike vs normal comparison plot (Range-Azimuth + NLCD)"""
    
    time_str = spike_info['time_str']
    
    from matplotlib.gridspec import GridSpec
    
    fig = plt.figure(figsize=(18, 14))
    gs = GridSpec(3, 2, figure=fig, height_ratios=[1, 1, 0.1], hspace=0.3, wspace=0.25)
    
    # =========================================================================
    # Row 0: NORMAL (Classification + NLCD)
    # Row 1: SPIKE (Classification + NLCD)
    # =========================================================================
    for row, (data, info, label) in enumerate([
        (normal_data, normal_info, 'NORMAL'),
        (spike_data, spike_info, 'SPIKE')
    ]):
        heading = info['heading']
        error = info['error']
        pass_num = get_pass_from_filename(info['group_key']) if 'group_key' in info else 'UNK'
        tile = info['tile']
        sat_info = info['sat_info']
        
        sel = data['selected_mask']
        unsel = ~data['selected_mask']
        
        color = 'green' if label == 'NORMAL' else 'red'
        
        # ---------------------------------------------------------------------
        # Left: PIXC Classification (Range-Azimuth)
        # ---------------------------------------------------------------------
        ax1 = fig.add_subplot(gs[row, 0])
        
        if sel.sum() > 0:
            ax1.scatter(data['range'][sel], data['azimuth'][sel], c=data['classification'][sel],
                       cmap=ListedColormap(CLASSIFICATION_COLORS), s=3, alpha=0.8,
                       marker='x', linewidths=0.8, vmin=1, vmax=7)
        if unsel.sum() > 0:
            ax1.scatter(data['range'][unsel], data['azimuth'][unsel], c=data['classification'][unsel],
                       cmap=ListedColormap(CLASSIFICATION_COLORS), s=3, alpha=0.4,
                       marker='o', vmin=1, vmax=7, edgecolors='none')
        
        ax1.set_xlabel('Range Index', fontsize=12, fontweight='bold')
        ax1.set_ylabel('Azimuth Index', fontsize=12, fontweight='bold')
        ax1.tick_params(axis='both', labelsize=10)
        ax1.set_title(f'{label}: PIXC Classification (Radar Coordinates)\n{heading:.0f}°-{tile} | Error: {error:.2f}m',
                      fontsize=12, fontweight='bold', color=color)
        ax1.grid(True, alpha=0.3)
        
        add_satellite_and_path(ax1, data['range'], data['azimuth'], heading, sat_info, pass_num, tile)
        
        for spine in ax1.spines.values():
            spine.set_linewidth(2)
            spine.set_color(color)
        
        # ---------------------------------------------------------------------
        # Right: NLCD (Lon-Lat)
        # ---------------------------------------------------------------------
        ax2 = fig.add_subplot(gs[row, 1])
        
        buffer = 0.005
        with rasterio.open(nlcd_path) as src:
            window = src.window(bounds[0]-buffer, bounds[1]-buffer, bounds[2]+buffer, bounds[3]+buffer)
            nlcd_data_arr = src.read(1, window=window)
            unique_vals = np.unique(nlcd_data_arr[nlcd_data_arr > 0])
            extent = [bounds[0]-buffer, bounds[2]+buffer, bounds[1]-buffer, bounds[3]+buffer]
            
            nlcd_cmap_colors = ['#FFFFFF'] * 256
            for val, clr in NLCD_COLORS.items():
                nlcd_cmap_colors[val] = clr
            nlcd_cmap = ListedColormap(nlcd_cmap_colors)
            
            ax2.imshow(nlcd_data_arr, extent=extent, cmap=nlcd_cmap,
                      vmin=0, vmax=255, interpolation='nearest')
        
        polygon.boundary.plot(ax=ax2, color=POLYGON_COLOR, linewidth=2.5)
        ax2.set_xlim(bounds[0]-buffer, bounds[2]+buffer)
        ax2.set_ylim(bounds[1]-buffer, bounds[3]+buffer)
        ax2.set_aspect('equal')
        
        ax2.xaxis.set_major_formatter(FuncFormatter(lon_formatter))
        ax2.yaxis.set_major_formatter(FuncFormatter(lat_formatter))
        ax2.set_xlabel('Longitude', fontsize=12, fontweight='bold')
        ax2.set_ylabel('Latitude', fontsize=12, fontweight='bold')
        ax2.tick_params(axis='both', labelsize=10)
        plt.setp(ax2.xaxis.get_majorticklabels(), rotation=15, ha='right')
        
        ax2.set_title(f'{label}: Land Cover (NLCD 2024)\n{info["time_str"]}',
                      fontsize=12, fontweight='bold', color=color)
        ax2.grid(True, alpha=0.3)
        
        for spine in ax2.spines.values():
            spine.set_linewidth(2)
            spine.set_color(color)
    
    # =========================================================================
    # Row 2: Legends
    # =========================================================================
    ax_leg1 = fig.add_subplot(gs[2, 0])
    ax_leg1.axis('off')
    pixc_legend = [Patch(facecolor=CLASSIFICATION_COLORS[i], edgecolor='black',
                         linewidth=0.5, label=CLASSIFICATION_LABELS[i]) for i in range(7)]
    ax_leg1.legend(handles=pixc_legend, loc='center', ncol=4, fontsize=9,
                   frameon=True, fancybox=True, edgecolor='gray',
                   title='SWOT PIXC Classification', title_fontsize=10)
    
    ax_leg2 = fig.add_subplot(gs[2, 1])
    ax_leg2.axis('off')
    nlcd_legend = [Patch(facecolor=NLCD_COLORS[val], edgecolor='black',
                         linewidth=0.5, label=NLCD_LABELS[val])
                   for val in sorted(unique_vals) if val in NLCD_COLORS]
    ax_leg2.legend(handles=nlcd_legend, loc='center', ncol=4, fontsize=8,
                   frameon=True, fancybox=True, edgecolor='gray',
                   title='NLCD Land Cover Classes', title_fontsize=10)
    
    # Main title
    fig.suptitle(f'ISA = {fctimp_str} ({float(fctimp_str)*100:.0f}%) | Spike vs Normal Comparison',
                 fontsize=16, fontweight='bold', y=0.98)
    
    # Save
    save_path = f'{output_dir}/COMPARE_ISA{fctimp_str}_{time_str}.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"  ✓ Saved: {save_path}")
    
    plt.close()

In [ ]:
process_fctimp('0.06', spike_threshold=0.1832, output_dir='figures', year=2025)

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch, Polygon as MplPolygon
from matplotlib.ticker import FuncFormatter
from matplotlib.gridspec import GridSpec
import rasterio
from pathlib import Path
import xarray as xr
from shapely.geometry import Point
from datetime import datetime, timedelta
import re

# ============================================================================
# Helper functions
# ============================================================================

def filter_best_pixc_files(files):
    file_dict = {}
    for f in files:
        match = re.match(r'(.+)_(P[IG]D\d)_(\d+)\.nc', f.name)
        if not match:
            continue
        base, processor, counter = match.group(1), match.group(2), int(match.group(3))
        if base not in file_dict:
            file_dict[base] = (f, processor, counter)
        else:
            _, old_proc, old_cnt = file_dict[base]
            if (processor == 'PGD0' and old_proc == 'PID0') or (processor == old_proc and counter > old_cnt):
                file_dict[base] = (f, processor, counter)
    return [f for f, _, _ in file_dict.values()]


def parse_filename_info(filename):
    parts = filename.split('_')
    if len(parts) >= 7:
        cycle_id = parts[4]
        pass_id = parts[5]
        tile_id = parts[6]
        return cycle_id, pass_id, tile_id
    return 'UNK', 'UNK', 'UNK'


def match_and_group_pixc(time_str, files, crid, window_minutes=5):
    dt = pd.to_datetime(time_str)
    if dt.tzinfo:
        dt = dt.tz_localize(None)
    start, end = dt - timedelta(minutes=window_minutes), dt + timedelta(minutes=window_minutes)
    groups = {}
    for f in files:
        parts = f.name.split('_')
        if len(parts) >= 10:
            try:
                fs, fe = pd.to_datetime(parts[7], format='%Y%m%dT%H%M%S'), pd.to_datetime(parts[8], format='%Y%m%dT%H%M%S')
                if not (fe < start or fs > end) and parts[9] == crid:
                    key = f"{parts[7]}_{parts[8]}"
                    groups.setdefault(key, []).append(f)
            except:
                continue
    return groups


def find_matching_pixcvec(pixc_file, pixcvec_files):
    pixc_name = pixc_file.name.replace('PIXC_', 'PIXCVec_')
    for pvf in pixcvec_files:
        if pvf.name == pixc_name:
            return pvf
    return None


def extract_and_merge_pixels(pixc_files, pixcvec_files, polygon_gdf):
    all_data = [extract_pixels_with_vec(pf, pvf, polygon_gdf) for pf, pvf in zip(pixc_files, pixcvec_files)]
    all_data = [d for d in all_data if d]
    if not all_data:
        return None
    if len(all_data) == 1:
        return all_data[0]
    merged = {k: np.concatenate([d[k] for d in all_data]) for k in all_data[0].keys() 
              if k not in ['heading', 'sat_info']}
    merged['heading'] = np.nanmean([d['heading'] for d in all_data])
    merged['sat_info'] = all_data[0]['sat_info']
    return merged


def extract_pixels_with_vec(pixc_file, pixcvec_file, polygon_gdf):
    ds = xr.open_dataset(pixc_file, group='pixel_cloud')
    lon, lat = ds['longitude'].values, ds['latitude'].values
    bounds = polygon_gdf.total_bounds
    
    ds_tvp = xr.open_dataset(pixc_file, group='tvp')
    heading = np.nanmedian(ds_tvp['velocity_heading'].values)
    sat_lon = np.nanmedian(ds_tvp['longitude'].values)
    sat_lat = np.nanmedian(ds_tvp['latitude'].values)
    sat_alt = np.nanmedian(ds_tvp['altitude'].values)
    ds_tvp.close()
    
    bbox_mask = ((lon >= bounds[0]) & (lon <= bounds[2]) & (lat >= bounds[1]) & (lat <= bounds[3]))
    if bbox_mask.sum() == 0:
        ds.close()
        return None
    
    classification = ds['classification'].values[bbox_mask]
    water_mask = classification > 2
    if water_mask.sum() == 0:
        ds.close()
        return None
    
    lon_filt, lat_filt = lon[bbox_mask][water_mask], lat[bbox_mask][water_mask]
    poly = polygon_gdf.geometry.iloc[0]
    inside_mask = np.array([poly.contains(Point(x, y)) for x, y in zip(lon_filt, lat_filt)])
    if inside_mask.sum() == 0:
        ds.close()
        return None
    
    final_idx = np.where(bbox_mask)[0][water_mask][inside_mask]
    
    cross_track = ds['cross_track'].values[final_idx]
    median_cross_track = np.nanmedian(cross_track)
    
    ds_vec = xr.open_dataset(pixcvec_file)
    reach_id = ds_vec['reach_id'].values[final_idx]
    node_id = ds_vec['node_id'].values[final_idx]
    obs_id = ds_vec['obs_id'].values[final_idx]
    lake_id = ds_vec['lake_id'].values[final_idx]
    
    def is_valid(arr):
        return np.array([len(x) > 0 and x != b'' for x in arr]) if arr.dtype.kind == 'S' else arr > 0
    
    selected_mask = is_valid(reach_id) & is_valid(node_id) & ~is_valid(obs_id) & ~is_valid(lake_id)
    
    result = {
        'lon': lon[final_idx], 'lat': lat[final_idx],
        'height': ds['height'].values[final_idx],
        'classification': ds['classification'].values[final_idx],
        'azimuth': ds['azimuth_index'].values[final_idx],
        'range': ds['range_index'].values[final_idx],
        'cross_track': cross_track,
        'selected_mask': selected_mask,
        'heading': heading,
        'sat_info': {
            'lon': sat_lon,
            'lat': sat_lat,
            'altitude': sat_alt,
            'cross_track': median_cross_track
        }
    }
    ds.close()
    ds_vec.close()
    return result


# ============================================================================
# Constants
# ============================================================================

CLASSIFICATION_COLORS = [
    'maroon', 'gold', 'skyblue', 'mediumblue', 'dimgray', 'cyan', 'darkturquoise'
]

CLASSIFICATION_LABELS = [
    'Land', 'Land (near water)', 'Water (near land)', 'Open Water',
    'Dark Water', 'Low Coh. (near land)', 'Low Coh. (open)'
]

NLCD_COLORS = {
    11: '#476BA1', 12: '#D1DDF9', 21: '#DDC9C9', 22: '#D89382',
    23: '#ED0000', 24: '#AA0000', 31: '#B2ADA3', 41: '#68AA63',
    42: '#1C6330', 43: '#B5C98E', 52: '#CCBA7C', 71: '#E2E2C1',
    81: '#DDD83E', 82: '#AE7229', 90: '#BBD7ED', 95: '#71A4C1'
}

NLCD_LABELS = {
    11: 'Open Water', 12: 'Ice/Snow', 21: 'Developed (Open)', 22: 'Developed (Low)',
    23: 'Developed (Med)', 24: 'Developed (High)', 31: 'Barren', 41: 'Deciduous Forest',
    42: 'Evergreen Forest', 43: 'Mixed Forest', 52: 'Shrub/Scrub', 71: 'Grassland',
    81: 'Pasture/Hay', 82: 'Cultivated', 90: 'Woody Wetlands', 95: 'Herbaceous Wetlands'
}

POLYGON_COLOR = '#404040'


def lon_formatter(x, pos):
    if x < 0:
        return f'{abs(x):.2f}°W'
    elif x > 0:
        return f'{x:.2f}°E'
    else:
        return '0°'

def lat_formatter(y, pos):
    if y > 0:
        return f'{y:.2f}°N'
    elif y < 0:
        return f'{abs(y):.2f}°S'
    else:
        return '0°'


def get_satellite_side(heading, cross_track):
    """위성이 왼쪽에서 오는지 오른쪽에서 오는지 결정"""
    if (0 <= heading < 90) or (270 <= heading < 360):
        return 'left' if cross_track >= 0 else 'right'
    else:
        return 'right' if cross_track >= 0 else 'left'


def draw_satellite_swath_range(ax, side, heading, cycle_id, pass_id, tile_id, 
                                x_min, x_max, y_min, y_max, band_width, text_color='black'):
    """
    Classification plot에 기울어진 위성 경로 band 그리기
    heading에 따라 14도 또는 166도 기울기
    """
    # 북위 기준 각도
    going_north = (0 <= heading < 90) or (270 <= heading < 360)
    angle = 14 if going_north else 166
    angle_rad = np.radians(angle)
    
    # band 중심 x 위치
    if side == 'left':
        center_x = x_min + band_width / 2
    else:
        center_x = x_max - band_width / 2
    
    # 기울기에 따른 offset 계산
    y_span = y_max - y_min
    x_offset = y_span * np.tan(np.radians(angle - 90)) * 0.3  # 살짝 기울기
    
    # 4개 꼭지점 (기울어진 평행사변형)
    if going_north:  # 14도 - 위로 갈수록 오른쪽으로
        points = [
            (center_x - band_width/2 - x_offset, y_min),
            (center_x + band_width/2 - x_offset, y_min),
            (center_x + band_width/2 + x_offset, y_max),
            (center_x - band_width/2 + x_offset, y_max),
        ]
    else:  # 166도 - 아래로 갈수록 오른쪽으로
        points = [
            (center_x - band_width/2 + x_offset, y_min),
            (center_x + band_width/2 + x_offset, y_min),
            (center_x + band_width/2 - x_offset, y_max),
            (center_x - band_width/2 - x_offset, y_max),
        ]
    
    band_patch = MplPolygon(points, facecolor='#505050', edgecolor='#404040', 
                             alpha=0.75, linewidth=1, zorder=3)
    ax.add_patch(band_patch)
    
    # 텍스트 위치 (band 중앙)
    text_x = center_x
    text_y_center = (y_min + y_max) / 2
    text_spacing = y_span * 0.12
    text_size = 12
    
    ax.text(text_x, text_y_center + text_spacing, f'Tile: {tile_id}', fontsize=text_size, fontweight='bold',
            color=text_color, ha='center', va='center', zorder=5)
    ax.text(text_x, text_y_center, f'Pass: {pass_id}', fontsize=text_size, fontweight='bold',
            color=text_color, ha='center', va='center', zorder=5)
    ax.text(text_x, text_y_center - text_spacing, f'Cycle: {cycle_id}', fontsize=text_size, fontweight='bold',
            color=text_color, ha='center', va='center', zorder=5)
    
    # Heading 각도 표시 (band 위 또는 아래)
    if going_north:
        ax.text(text_x, y_max + y_span * 0.03, f'{heading:.0f}°', fontsize=text_size + 1, fontweight='bold',
                color=text_color, ha='center', va='bottom', zorder=5)
    else:
        ax.text(text_x, y_min - y_span * 0.03, f'{heading:.0f}°', fontsize=text_size + 1, fontweight='bold',
                color=text_color, ha='center', va='top', zorder=5)


def draw_satellite_swath_lonlat(ax, side, heading, cycle_id, pass_id, tile_id,
                                 x_min, x_max, y_min, y_max, band_width, text_color='white'):
    """
    NLCD plot에 기울어진 위성 경로 band 그리기
    텍스트는 안쪽으로 (짤리지 않게)
    """
    going_north = (0 <= heading < 90) or (270 <= heading < 360)
    
    if side == 'left':
        center_x = x_min + band_width / 2
    else:
        center_x = x_max - band_width / 2
    
    y_span = y_max - y_min
    x_offset = y_span * 0.15
    
    if going_north:
        points = [
            (center_x - band_width/2 - x_offset * 0.3, y_min),
            (center_x + band_width/2 - x_offset * 0.3, y_min),
            (center_x + band_width/2 + x_offset * 0.3, y_max),
            (center_x - band_width/2 + x_offset * 0.3, y_max),
        ]
    else:
        points = [
            (center_x - band_width/2 + x_offset * 0.3, y_min),
            (center_x + band_width/2 + x_offset * 0.3, y_min),
            (center_x + band_width/2 - x_offset * 0.3, y_max),
            (center_x - band_width/2 - x_offset * 0.3, y_max),
        ]
    
    band_patch = MplPolygon(points, facecolor='#505050', edgecolor='#404040', 
                             alpha=0.75, linewidth=1, zorder=3)
    ax.add_patch(band_patch)
    
    # 텍스트 위치 - 더 안쪽으로 (band 중심에서 안쪽으로 offset)
    if side == 'left':
        text_x = center_x + band_width * 0.15  # 오른쪽으로
    else:
        text_x = center_x - band_width * 0.15  # 왼쪽으로
    
    text_y_center = (y_min + y_max) / 2
    text_spacing = y_span * 0.10  # 간격 줄임
    text_size = 11  # 글씨 크기 약간 줄임
    
    ax.text(text_x, text_y_center + text_spacing, f'Tile: {tile_id}', fontsize=text_size, fontweight='bold',
            color=text_color, ha='center', va='center', zorder=5)
    ax.text(text_x, text_y_center, f'Pass: {pass_id}', fontsize=text_size, fontweight='bold',
            color=text_color, ha='center', va='center', zorder=5)
    ax.text(text_x, text_y_center - text_spacing, f'Cycle: {cycle_id}', fontsize=text_size, fontweight='bold',
            color=text_color, ha='center', va='center', zorder=5)
    
    # Heading 각도 - band 안쪽 상단/하단
    if going_north:
        ax.text(text_x, y_max - y_span * 0.05, f'{heading:.0f}°', fontsize=text_size + 1, fontweight='bold',
                color=text_color, ha='center', va='top', zorder=5)
    else:
        ax.text(text_x, y_min + y_span * 0.05, f'{heading:.0f}°', fontsize=text_size + 1, fontweight='bold',
                color=text_color, ha='center', va='bottom', zorder=5)


# ============================================================================
# Main function
# ============================================================================

def process_fctimp(fctimp_str, spike_threshold=0.1832, output_dir=None, year=2025):
    if output_dir is None:
        output_dir = f'isa_{fctimp_str}_v2'
    Path(output_dir).mkdir(exist_ok=True)
    
    selected_12 = gpd.read_file('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/selected_12_vD.shp')
    fctimp_val = float(fctimp_str)
    sample = selected_12[np.isclose(selected_12["fctimp_mea"], fctimp_val, atol=0.01)].copy()
    if sample.empty:
        print(f"No reach for {fctimp_str}")
        return
    
    df = pd.read_parquet(f'/work/pi_kandread_umass_edu/swot-urban/1_data_processing/fctimp_mean_{fctimp_str}_sample.parquet')
    d = df.copy()
    d['time_dt'] = pd.to_datetime(d['time_str'])
    d = d.sort_values('time_dt').reset_index(drop=True)
    d = d[d['time_dt'].dt.year == year].reset_index(drop=True)
    
    if 'error_relative_m' in d.columns:
        d['error'] = d['error_relative_m']
    else:
        d['error'] = d['wse_relative_m'] - d['wse_usgs_relative_m']
    
    d['is_spike'] = d['error'].abs() >= spike_threshold
    
    print(f"ISA = {fctimp_str} ({year}): {len(d)} observations")
    print(f"  High Error (|error| >= {spike_threshold}m): {d['is_spike'].sum()}")
    print(f"  Low Error (|error| < {spike_threshold}m): {(~d['is_spike']).sum()}")
    
    pixc_dir = Path(f'/project/pi_kandread_umass_edu/swot-urban/0_data_collection/SWOT_PIXC/fctimp_{fctimp_str}')
    pixcvec_dir = Path(f'/project/pi_kandread_umass_edu/swot-urban/0_data_collection/SWOT_PIXCVec/fctimp_{fctimp_str}')
    pixc_files = filter_best_pixc_files(list(pixc_dir.glob('SWOT_L2_HR_PIXC_*.nc')))
    pixcvec_files = filter_best_pixc_files(list(pixcvec_dir.glob('SWOT_L2_HR_PIXCVec_*.nc')))
    
    print(f"  PIXC files: {len(pixc_files)}")
    
    polygon = sample.to_crs('EPSG:4326')
    bounds = polygon.total_bounds
    nlcd_path = '/work/pi_kandread_umass_edu/swot-urban/0_data_collection/1_nlcd/Annual_NLCD_LndCov_2024_CU_C1V1_EPSG4326.tif'
    
    for idx, spike_row in d[d['is_spike']].iterrows():
        print(f"\n{'='*70}")
        print(f"High Error: {spike_row['time_str']} (error={spike_row['error']:.2f}m)")
        
        normals = d[~d['is_spike']].copy()
        if len(normals) == 0:
            continue
        normals['time_diff'] = (normals['time_dt'] - spike_row['time_dt']).abs()
        closest_normal = normals.loc[normals['time_diff'].idxmin()]
        print(f"Low Error: {closest_normal['time_str']} (error={closest_normal['error']:.2f}m)")
        
        # Get Low Error data
        normal_data = None
        normal_info = None
        for group_key, nf_list in match_and_group_pixc(closest_normal['time_str'], pixc_files, closest_normal['crid']).items():
            vec_list = [v for v in [find_matching_pixcvec(nf, pixcvec_files) for nf in nf_list] if v]
            if not vec_list:
                continue
            data = extract_and_merge_pixels(nf_list, vec_list, polygon)
            if data:
                cycle_id, pass_id, tile_id = parse_filename_info(nf_list[0].name)
                normal_data = data
                normal_info = {
                    'time_str': closest_normal['time_dt'].strftime('%Y%m%dT%H%M%S'),
                    'error': closest_normal['error'],
                    'cycle_id': cycle_id,
                    'pass_id': pass_id,
                    'tile_id': tile_id,
                    'heading': data['heading'],
                    'sat_info': data['sat_info']
                }
                break
        
        # Get High Error data
        spike_data = None
        spike_info = None
        for group_key, sf_list in match_and_group_pixc(spike_row['time_str'], pixc_files, spike_row['crid']).items():
            vec_list = [v for v in [find_matching_pixcvec(sf, pixcvec_files) for sf in sf_list] if v]
            if not vec_list:
                continue
            data = extract_and_merge_pixels(sf_list, vec_list, polygon)
            if data:
                cycle_id, pass_id, tile_id = parse_filename_info(sf_list[0].name)
                spike_data = data
                spike_info = {
                    'time_str': spike_row['time_dt'].strftime('%Y%m%dT%H%M%S'),
                    'error': spike_row['error'],
                    'cycle_id': cycle_id,
                    'pass_id': pass_id,
                    'tile_id': tile_id,
                    'heading': data['heading'],
                    'sat_info': data['sat_info']
                }
                break
        
        if normal_data and spike_data:
            save_comparison_plot(normal_data, spike_data, normal_info, spike_info, 
                                 polygon, bounds, output_dir, fctimp_str, nlcd_path)
        else:
            if not normal_data:
                print("  → Could not extract Low Error data")
            if not spike_data:
                print("  → Could not extract High Error data")
    
    print(f"\n✓ Done: {output_dir}")


# ============================================================================
# Plot function
# ============================================================================

def save_comparison_plot(normal_data, spike_data, normal_info, spike_info, polygon, bounds, output_dir, fctimp_str, nlcd_path):
    time_str = spike_info['time_str']
    
    FONT_SIZE = 14
    LON_EXTEND = 0.06  # NLCD만 확장
    BAND_WIDTH_LON = 0.035
    
    fig = plt.figure(figsize=(22, 15))
    gs = GridSpec(3, 2, figure=fig, height_ratios=[1, 1, 0.18], 
                  hspace=0.35, wspace=0.3)
    
    unique_vals = None
    
    for row, (data, info, label) in enumerate([
        (normal_data, normal_info, 'Low Error'),
        (spike_data, spike_info, 'High Error')
    ]):
        heading = info['heading']
        error = info['error']
        cycle_id = info['cycle_id']
        pass_id = info['pass_id']
        tile_id = info['tile_id']
        cross_track = info['sat_info']['cross_track']
        
        sel = data['selected_mask']
        
        frame_color = 'green' if label == 'Low Error' else 'red'
        sat_side = get_satellite_side(heading, cross_track)
        
        # Range limits - 딱 맞게
        range_min, range_max = np.nanmin(data['range'][sel]), np.nanmax(data['range'][sel])
        az_min, az_max = np.nanmin(data['azimuth'][sel]), np.nanmax(data['azimuth'][sel])
        
        # =====================================================================
        # Left: SWOT Classification - 위성 shading 없이, Range 확장 없이
        # =====================================================================
        ax1 = fig.add_subplot(gs[row, 0])
        
        if sel.sum() > 0:
            ax1.scatter(data['range'][sel], data['azimuth'][sel], c=data['classification'][sel],
                       cmap=ListedColormap(CLASSIFICATION_COLORS), s=1, alpha=0.8,
                       marker='o', vmin=1, vmax=7, edgecolors='none')
        
        # 딱 맞게 (약간의 여백만)
        range_margin = (range_max - range_min) * 0.02
        az_margin = (az_max - az_min) * 0.02
        ax1.set_xlim(range_min - range_margin, range_max + range_margin)
        ax1.set_ylim(az_min - az_margin, az_max + az_margin)
        
        ax1.set_xlabel('Range', fontsize=FONT_SIZE, fontweight='bold')
        ax1.set_ylabel('Azimuth', fontsize=FONT_SIZE, fontweight='bold')
        ax1.tick_params(axis='both', labelsize=FONT_SIZE - 2)
        
        ax1.set_title(f'SWOT Classification (Error: {error:.2f}m)', 
                      fontsize=FONT_SIZE + 2, fontweight='bold', color=frame_color, pad=15)
        ax1.grid(True, alpha=0.3)
        
        for spine in ax1.spines.values():
            spine.set_linewidth(2.5)
            spine.set_color(frame_color)
        
        # 위성 shading 안 그림!
        
        # =====================================================================
        # Right: NLCD Land Cover
        # =====================================================================
        ax2 = fig.add_subplot(gs[row, 1])
        
        buffer = 0.008
        lon_min, lon_max = bounds[0] - buffer, bounds[2] + buffer
        lat_min, lat_max = bounds[1] - buffer, bounds[3] + buffer
        lat_span = lat_max - lat_min
        
        # xlim 확장 (NLCD만)
        if sat_side == 'left':
            plot_lon_min = lon_min - LON_EXTEND
            plot_lon_max = lon_max
        else:
            plot_lon_min = lon_min
            plot_lon_max = lon_max + LON_EXTEND
        
        plot_lat_min = lat_min - lat_span * 0.08
        plot_lat_max = lat_max + lat_span * 0.08
        
        with rasterio.open(nlcd_path) as src:
            window = src.window(plot_lon_min, plot_lat_min, plot_lon_max, plot_lat_max)
            nlcd_data_arr = src.read(1, window=window)
            unique_vals = np.unique(nlcd_data_arr[nlcd_data_arr > 0])
            extent = [plot_lon_min, plot_lon_max, plot_lat_min, plot_lat_max]
            
            nlcd_cmap_colors = ['#FFFFFF'] * 256
            for val, clr in NLCD_COLORS.items():
                nlcd_cmap_colors[val] = clr
            nlcd_cmap = ListedColormap(nlcd_cmap_colors)
            
            ax2.imshow(nlcd_data_arr, extent=extent, cmap=nlcd_cmap,
                      vmin=0, vmax=255, interpolation='nearest')
        
        polygon.boundary.plot(ax=ax2, color=POLYGON_COLOR, linewidth=2.5)
        
        ax2.set_xlim(plot_lon_min, plot_lon_max)
        ax2.set_ylim(plot_lat_min, plot_lat_max)
        ax2.set_aspect('equal')
        
        ax2.xaxis.set_major_formatter(FuncFormatter(lon_formatter))
        ax2.yaxis.set_major_formatter(FuncFormatter(lat_formatter))
        ax2.set_xlabel('Longitude', fontsize=FONT_SIZE, fontweight='bold')
        ax2.set_ylabel('Latitude', fontsize=FONT_SIZE, fontweight='bold')
        ax2.tick_params(axis='both', labelsize=FONT_SIZE - 2)
        plt.setp(ax2.xaxis.get_majorticklabels(), rotation=15, ha='right')
        
        ax2.set_title(f'NLCD Land Cover (Error: {error:.2f}m)', 
                      fontsize=FONT_SIZE + 2, fontweight='bold', color=frame_color, pad=15)
        ax2.grid(True, alpha=0.3)
        
        for spine in ax2.spines.values():
            spine.set_linewidth(2.5)
            spine.set_color(frame_color)
        
        # 위성 경로 band 그리기 (NLCD만, 글씨 안쪽으로)
        draw_satellite_swath_lonlat(ax2, sat_side, heading, cycle_id, pass_id, tile_id,
                                     plot_lon_min, plot_lon_max, plot_lat_min, plot_lat_max, 
                                     BAND_WIDTH_LON, text_color='white')
    
    # =========================================================================
    # Legends
    # =========================================================================
    ax_leg1 = fig.add_subplot(gs[2, 0])
    ax_leg1.axis('off')
    pixc_legend = [Patch(facecolor=CLASSIFICATION_COLORS[i], edgecolor='black',
                         linewidth=0.5, label=CLASSIFICATION_LABELS[i]) for i in range(7)]
    ax_leg1.legend(handles=pixc_legend, loc='center', ncol=4, fontsize=FONT_SIZE - 1,
                   frameon=True, fancybox=True, edgecolor='gray',
                   title='PIXC Classification', title_fontsize=FONT_SIZE,
                   columnspacing=1, handletextpad=0.5)
    
    ax_leg2 = fig.add_subplot(gs[2, 1])
    ax_leg2.axis('off')
    if unique_vals is not None:
        nlcd_legend = [Patch(facecolor=NLCD_COLORS[val], edgecolor='black',
                             linewidth=0.5, label=NLCD_LABELS[val])
                       for val in sorted(unique_vals) if val in NLCD_COLORS]
        ax_leg2.legend(handles=nlcd_legend, loc='center', ncol=4, fontsize=FONT_SIZE - 2,
                       frameon=True, fancybox=True, edgecolor='gray',
                       title='NLCD Land Cover', title_fontsize=FONT_SIZE,
                       columnspacing=0.8, handletextpad=0.4)
    
    save_path = f'{output_dir}/figure4_COMPARE_ISA{fctimp_str}_{time_str}.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"  ✓ Saved: {save_path}")
    plt.close()

# 실행
process_fctimp('0.06', spike_threshold=0.1832, output_dir='figures', year=2025)

#### all

In [ ]:
def filter_best_pixc_files(files):
    file_dict = {}
    for f in files:
        match = re.match(r'(.+)_(P[IG]D\d)_(\d+)\.nc', f.name)
        if not match:
            continue
        base, processor, counter = match.group(1), match.group(2), int(match.group(3))
        if base not in file_dict:
            file_dict[base] = (f, processor, counter)
        else:
            _, old_proc, old_cnt = file_dict[base]
            if (processor == 'PGD0' and old_proc == 'PID0') or (processor == old_proc and counter > old_cnt):
                file_dict[base] = (f, processor, counter)
    return [f for f, _, _ in file_dict.values()]

def get_tile_from_filename(filename):
    parts = filename.split('_')
    if len(parts) >= 7:
        return parts[6]
    return 'UNK'

def get_heading_from_pixc(pixc_file):
    try:
        ds_tvp = xr.open_dataset(pixc_file, group='tvp')
        heading = np.nanmedian(ds_tvp['velocity_heading'].values)
        ds_tvp.close()
        return heading
    except:
        return None

def calculate_sun_position(dt, lat, lon):
    """
    Calculate sun azimuth and elevation for given datetime and location
    
    Returns:
        azimuth: degrees from North (0=N, 90=E, 180=S, 270=W)
        elevation: degrees above horizon
    """
    # Handle pandas Timestamp
    if hasattr(dt, 'to_pydatetime'):
        dt = dt.to_pydatetime()
    
    # Handle timezone - convert to naive datetime (assuming UTC)
    if hasattr(dt, 'tzinfo') and dt.tzinfo is not None:
        # Remove timezone info (data is already in UTC)
        dt = dt.replace(tzinfo=None)
    
    # Days since J2000.0
    n = (dt - datetime(2000, 1, 1, 12, 0, 0)).total_seconds() / 86400.0
    
    # Mean longitude of the Sun
    L = (280.460 + 0.9856474 * n) % 360
    
    # Mean anomaly of the Sun
    g = math.radians((357.528 + 0.9856003 * n) % 360)
    
    # Ecliptic longitude
    lambda_sun = math.radians(L + 1.915 * math.sin(g) + 0.020 * math.sin(2 * g))
    
    # Obliquity of the ecliptic
    epsilon = math.radians(23.439 - 0.0000004 * n)
    
    # Right ascension and declination
    alpha = math.atan2(math.cos(epsilon) * math.sin(lambda_sun), math.cos(lambda_sun))
    delta = math.asin(math.sin(epsilon) * math.sin(lambda_sun))
    
    # Greenwich Mean Sidereal Time
    GMST = (18.697374558 + 24.06570982441908 * n) % 24
    
    # Local Sidereal Time
    LST = (GMST + lon / 15.0) % 24
    
    # Hour angle
    H = math.radians((LST * 15) - math.degrees(alpha))
    
    # Convert lat to radians
    phi = math.radians(lat)
    
    # Altitude (elevation)
    sin_alt = math.sin(phi) * math.sin(delta) + math.cos(phi) * math.cos(delta) * math.cos(H)
    elevation = math.degrees(math.asin(sin_alt))
    
    # Azimuth
    cos_az = (math.sin(delta) - math.sin(phi) * sin_alt) / (math.cos(phi) * math.cos(math.asin(sin_alt)))
    cos_az = max(-1, min(1, cos_az))  # Clamp to [-1, 1]
    azimuth = math.degrees(math.acos(cos_az))
    
    if math.sin(H) > 0:
        azimuth = 360 - azimuth
    
    return azimuth, elevation

def add_arrows_and_indicators(ax, bounds, heading, sun_azimuth=None, sun_elevation=None, sat_info=None):
    """
    Add North arrow, Satellite position with heading arrow, and Sun position
    - SAT: position based on cross-track sign, arrow shows heading direction
    - SUN: position based on azimuth, arrow points toward plot center
    - N: fixed at top as reference
    """
    # 1. NORTH ARROW - fixed at top right outside
    ax.annotate('', xy=(1.06, 1.08), xytext=(1.06, 0.98),
                xycoords='axes fraction', textcoords='axes fraction',
                arrowprops=dict(arrowstyle='->', color='black', lw=1.5),
                annotation_clip=False, zorder=10)
    ax.text(1.06, 1.10, 'N', transform=ax.transAxes,
            ha='center', va='bottom', fontsize=9, fontweight='bold', color='black', clip_on=False)
    
    # 2. SATELLITE - position based on cross-track AND heading direction
    if sat_info is not None:
        sat_lon = sat_info['lon']
        sat_lat = sat_info['lat']
        sat_alt = sat_info['altitude']
        xtrk = sat_info['cross_track']
        
        # Position based on heading and cross-track
        # Heading 0-90 or 270-360: northward component
        # Heading 90-270: southward component
        if (0 <= heading < 90) or (270 <= heading < 360):
            # Northward: xtrk>0 means sat is on LEFT (west)
            if xtrk >= 0:
                sat_x, sat_y = -0.12, 0.5  # LEFT
            else:
                sat_x, sat_y = 1.12, 0.5   # RIGHT
        else:  # 90 <= heading < 270
            # Southward: xtrk>0 means sat is on RIGHT (east)
            if xtrk >= 0:
                sat_x, sat_y = 1.12, 0.5   # RIGHT
            else:
                sat_x, sat_y = -0.12, 0.5  # LEFT
        
        # SAT box
        ax.plot(sat_x, sat_y, 's', color='blue', markersize=12, transform=ax.transAxes,
                markeredgecolor='white', markeredgewidth=1, clip_on=False, zorder=15)
        ax.text(sat_x, sat_y, 'SAT', transform=ax.transAxes,
                fontsize=6, ha='center', va='center', color='white', fontweight='bold', clip_on=False, zorder=16)
        
        # Heading arrow FROM satellite box
        arrow_len = 0.12
        heading_rad = np.radians(heading)
        dx = arrow_len * np.sin(heading_rad)
        dy = arrow_len * np.cos(heading_rad)
        
        ax.annotate('', xy=(sat_x + dx, sat_y + dy), xytext=(sat_x, sat_y),
                    xycoords='axes fraction', textcoords='axes fraction',
                    arrowprops=dict(arrowstyle='->', color='blue', lw=1.5),
                    annotation_clip=False, zorder=14)
        
        # Heading label - position based on arrow direction
        label_x = sat_x + dx * 1.4
        label_y = sat_y + dy * 1.4
        ax.text(label_x, label_y, f'{heading:.0f}°', transform=ax.transAxes,
                ha='center', va='center', fontsize=8, fontweight='bold', color='blue', clip_on=False)
        
        # SAT info text - below the box with enough spacing
        info_text = f'Lon:{sat_lon:.2f}°\nLat:{sat_lat:.2f}°\nAlt:{sat_alt/1000:.0f}km\nXtrk:{xtrk/1000:.1f}km'
        ax.text(sat_x, sat_y - 0.10, info_text, transform=ax.transAxes,
                ha='center', va='top', fontsize=5, color='blue', clip_on=False,
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='blue', alpha=0.9, linewidth=0.5),
                zorder=15)
    
    # 3. SUN - position based on azimuth
    if sun_azimuth is not None and sun_elevation is not None:
        if sun_elevation > 0:  # Daytime
            # Determine sun position based on azimuth
            # azimuth: 0=N, 90=E, 180=S, 270=W
            if 45 <= sun_azimuth < 135:  # East (right)
                sun_x, sun_y = 1.12, 0.85
                arrow_dx, arrow_dy = -0.08, 0
                label_x, label_y = sun_x + 0.08, sun_y  # Label to the right
                label_ha, label_va = 'left', 'center'
            elif 135 <= sun_azimuth < 225:  # South (bottom)
                sun_x, sun_y = 0.5, -0.12
                arrow_dx, arrow_dy = 0, 0.08
                label_x, label_y = sun_x, sun_y - 0.08  # Label below
                label_ha, label_va = 'center', 'top'
            elif 225 <= sun_azimuth < 315:  # West (left)
                sun_x, sun_y = -0.12, 0.85
                arrow_dx, arrow_dy = 0.08, 0
                label_x, label_y = sun_x - 0.08, sun_y  # Label to the left
                label_ha, label_va = 'right', 'center'
            else:  # North (top): 315-360 or 0-45
                sun_x, sun_y = 0.5, 1.12
                arrow_dx, arrow_dy = 0, -0.08
                label_x, label_y = sun_x, sun_y + 0.08  # Label above
                label_ha, label_va = 'center', 'bottom'
            
            # SUN box (orange)
            ax.plot(sun_x, sun_y, 's', color='#FFA500', markersize=12, transform=ax.transAxes,
                    markeredgecolor='white', markeredgewidth=1, clip_on=False, zorder=15)
            ax.text(sun_x, sun_y, 'SUN', transform=ax.transAxes,
                    fontsize=5, ha='center', va='center', color='white', fontweight='bold', clip_on=False, zorder=16)
            
            # Arrow FROM sun toward center
            ax.annotate('', xy=(sun_x + arrow_dx, sun_y + arrow_dy), xytext=(sun_x, sun_y),
                        xycoords='axes fraction', textcoords='axes fraction',
                        arrowprops=dict(arrowstyle='->', color='#FFA500', lw=1.5),
                        annotation_clip=False, zorder=14)
            
            # Sun Az and El label - positioned away from box
            sun_label = f'Az:{sun_azimuth:.0f}° El:{sun_elevation:.0f}°'
            ax.text(label_x, label_y, sun_label, transform=ax.transAxes, 
                    ha=label_ha, va=label_va,
                    fontsize=6, fontweight='bold', color='#FFA500', clip_on=False)
        else:
            # Night - just show text
            ax.text(1.06, 0.85, 'Night', transform=ax.transAxes,
                    ha='center', va='center', fontsize=8, color='#555555', clip_on=False)

# Classification colors
CLASSIFICATION_COLORS = [
    'maroon',        # 1: land - 고동색
    'gold',          # 2: land_near_water - 진한 노랑색
    'skyblue',       # 3: water_near_land - 하늘색
    'mediumblue',    # 4: open_water - 새파란색
    'dimgray',       # 5: dark_water - 진한 회색
    'cyan',          # 6: low_coh_water_near_land - 옅은 하늘색
    'darkturquoise', # 7: open_low_coh_water - 옅은 새파란색
]

CLASSIFICATION_LABELS = [
    'land', 'land_near_water', 'water_near_land', 'open_water',
    'dark_water', 'low_coh_water_near_land', 'open_low_coh_water'
]

POLYGON_COLOR = '#404040'

def process_fctimp(fctimp_str, spike_threshold=1.0, output_dir=None):
    if output_dir is None:
        output_dir = f'isa_{fctimp_str}_v2'
    Path(output_dir).mkdir(exist_ok=True)
    
    selected_12 = gpd.read_file('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/selected_12_vD.shp')
    fctimp_val = float(fctimp_str)
    sample = selected_12[np.isclose(selected_12["fctimp_mean"], fctimp_val, atol=0.01)].copy()
    if sample.empty:
        print(f"No reach for {fctimp_str}")
        return
    
    df = pd.read_parquet(f'/work/pi_kandread_umass_edu/swot-urban/1_data_processing/fctimp_meann_{fctimp_str}_sample.parquet')
    d = df.copy()
    d['time_dt'] = pd.to_datetime(d['time_str'])
    d = d.sort_values('time_dt').reset_index(drop=True)
    d['error'] = d['wse_relative_m'] - d['wse_usgs_relative_m']  # + = overestimate, - = underestimate
    d['is_spike'] = d['error'] >= spike_threshold
    
    pixc_dir = Path(f'/project/pi_kandread_umass_edu/swot-urban/0_data_collection/SWOT_PIXC/fctimp_{fctimp_str}')
    pixcvec_dir = Path(f'/project/pi_kandread_umass_edu/swot-urban/0_data_collection/SWOT_PIXCVec/fctimp_{fctimp_str}')
    pixc_files = filter_best_pixc_files(list(pixc_dir.glob('SWOT_L2_HR_PIXC_*.nc')))
    pixcvec_files = filter_best_pixc_files(list(pixcvec_dir.glob('SWOT_L2_HR_PIXCVec_*.nc')))
    
    polygon = sample.to_crs('EPSG:4326')
    bounds = polygon.total_bounds
    
    for idx, spike_row in d[d['is_spike']].iterrows():
        print(f"\n{'='*70}")
        print(f"Spike: {spike_row['time_str']} (error={spike_row['error']:.2f}m)")
        
        normals = d[~d['is_spike']].copy()
        if len(normals) == 0:
            continue
        normals['time_diff'] = (normals['time_dt'] - spike_row['time_dt']).abs()
        closest_normal = normals.loc[normals['time_diff'].idxmin()]
        print(f"Normal: {closest_normal['time_str']} (error={closest_normal['error']:.2f}m, Δ={closest_normal['time_diff']})")
        
        # Get Normal data
        normal_data = None
        normal_info = None
        for group_key, nf_list in match_and_group_pixc(closest_normal['time_str'], pixc_files, closest_normal['crid']).items():
            vec_list = [v for v in [find_matching_pixcvec(nf, pixcvec_files) for nf in nf_list] if v]
            if not vec_list:
                continue
            data = extract_and_merge_pixels(nf_list, vec_list, polygon)
            if data:
                normal_data = data
                normal_info = {
                    'time_str': closest_normal['time_dt'].strftime('%Y%m%dT%H%M%S'),
                    'time_dt': closest_normal['time_dt'],
                    'crid': closest_normal['crid'],
                    'error': closest_normal['error'],
                    'tile': get_tile_from_filename(nf_list[0].name),
                    'heading': data['heading'],
                    'group_key': group_key,
                    'sat_info': data['sat_info']
                }
                break
        
        # Get Spike data
        spike_data = None
        spike_info = None
        for group_key, sf_list in match_and_group_pixc(spike_row['time_str'], pixc_files, spike_row['crid']).items():
            vec_list = [v for v in [find_matching_pixcvec(sf, pixcvec_files) for sf in sf_list] if v]
            if not vec_list:
                continue
            data = extract_and_merge_pixels(sf_list, vec_list, polygon)
            if data:
                spike_data = data
                spike_info = {
                    'time_str': spike_row['time_dt'].strftime('%Y%m%dT%H%M%S'),
                    'time_dt': spike_row['time_dt'],
                    'crid': spike_row['crid'],
                    'error': spike_row['error'],
                    'tile': get_tile_from_filename(sf_list[0].name),
                    'heading': data['heading'],
                    'group_key': group_key,
                    'sat_info': data['sat_info']
                }
                break
        
        # Plot Normal-Spike comparison if both exist
        if normal_data and spike_data:
            save_comparison_plots(normal_data, spike_data, normal_info, spike_info, 
                                  polygon, bounds, output_dir, fctimp_str)
    
    print(f"\n✓ Done: {output_dir}")

def match_and_group_pixc(time_str, files, crid, window_minutes=5):
    dt = pd.to_datetime(time_str)
    if dt.tzinfo:
        dt = dt.tz_localize(None)
    start, end = dt - timedelta(minutes=window_minutes), dt + timedelta(minutes=window_minutes)
    groups = {}
    for f in files:
        parts = f.name.split('_')
        if len(parts) >= 10:
            try:
                fs, fe = pd.to_datetime(parts[7], format='%Y%m%dT%H%M%S'), pd.to_datetime(parts[8], format='%Y%m%dT%H%M%S')
                if not (fe < start or fs > end) and parts[9] == crid:
                    key = f"{parts[7]}_{parts[8]}"
                    groups.setdefault(key, []).append(f)
            except:
                continue
    return groups

def find_matching_pixcvec(pixc_file, pixcvec_files):
    pixc_name = pixc_file.name.replace('PIXC_', 'PIXCVec_')
    for pvf in pixcvec_files:
        if pvf.name == pixc_name:
            return pvf
    return None

def extract_and_merge_pixels(pixc_files, pixcvec_files, polygon_gdf):
    all_data = [extract_pixels_with_vec(pf, pvf, polygon_gdf) for pf, pvf in zip(pixc_files, pixcvec_files)]
    all_data = [d for d in all_data if d]
    if not all_data:
        return None
    if len(all_data) == 1:
        return all_data[0]
    merged = {k: np.concatenate([d[k] for d in all_data]) for k in all_data[0].keys() 
              if k not in ['heading', 'sat_info']}
    merged['heading'] = np.nanmean([d['heading'] for d in all_data])
    # Use first satellite info (they should be very similar)
    merged['sat_info'] = all_data[0]['sat_info']
    return merged

def extract_pixels_with_vec(pixc_file, pixcvec_file, polygon_gdf):
    ds = xr.open_dataset(pixc_file, group='pixel_cloud')
    lon, lat = ds['longitude'].values, ds['latitude'].values
    bounds = polygon_gdf.total_bounds
    
    # Get tvp group data (satellite position)
    ds_tvp = xr.open_dataset(pixc_file, group='tvp')
    heading = np.nanmedian(ds_tvp['velocity_heading'].values)
    
    # Satellite position - use median values
    sat_lon = np.nanmedian(ds_tvp['longitude'].values)
    sat_lat = np.nanmedian(ds_tvp['latitude'].values)
    sat_alt = np.nanmedian(ds_tvp['altitude'].values)
    ds_tvp.close()
    
    bbox_mask = ((lon >= bounds[0]) & (lon <= bounds[2]) & (lat >= bounds[1]) & (lat <= bounds[3]))
    if bbox_mask.sum() == 0:
        ds.close()
        return None
    
    classification = ds['classification'].values[bbox_mask]
    water_mask = classification > 2
    if water_mask.sum() == 0:
        ds.close()
        return None
    
    lon_filt, lat_filt = lon[bbox_mask][water_mask], lat[bbox_mask][water_mask]
    poly = polygon_gdf.geometry.iloc[0]
    inside_mask = np.array([poly.contains(Point(x, y)) for x, y in zip(lon_filt, lat_filt)])
    if inside_mask.sum() == 0:
        ds.close()
        return None
    
    final_idx = np.where(bbox_mask)[0][water_mask][inside_mask]
    
    # Get cross-track distance
    cross_track = ds['cross_track'].values[final_idx]
    median_cross_track = np.nanmedian(cross_track)
    
    ds_vec = xr.open_dataset(pixcvec_file)
    reach_id = ds_vec['reach_id'].values[final_idx]
    node_id = ds_vec['node_id'].values[final_idx]
    obs_id = ds_vec['obs_id'].values[final_idx]
    lake_id = ds_vec['lake_id'].values[final_idx]
    
    def is_valid(arr):
        return np.array([len(x) > 0 and x != b'' for x in arr]) if arr.dtype.kind == 'S' else arr > 0
    
    selected_mask = is_valid(reach_id) & is_valid(node_id) & ~is_valid(obs_id) & ~is_valid(lake_id)
    
    result = {
        'lon': lon[final_idx], 'lat': lat[final_idx],
        'lon_all': lon, 'lat_all': lat,
        'height': ds['height'].values[final_idx],
        'classification': ds['classification'].values[final_idx],
        'bright_land': ds['bright_land_flag'].values[final_idx],
        'layover': ds['layover_impact'].values[final_idx],
        'sig0': ds['sig0'].values[final_idx],
        'coh_power': ds['coherent_power'].values[final_idx],
        'azimuth': ds['azimuth_index'].values[final_idx],
        'range': ds['range_index'].values[final_idx],
        'cross_track': cross_track,
        'selected_mask': selected_mask,
        'heading': heading,
        'sat_info': {
            'lon': sat_lon,
            'lat': sat_lat,
            'altitude': sat_alt,
            'cross_track': median_cross_track
        }
    }
    ds.close()
    ds_vec.close()
    return result

def save_comparison_plots(normal_data, spike_data, normal_info, spike_info, polygon, bounds, output_dir, fctimp_str):
    """Save side-by-side Normal vs Spike comparison plots"""
    
    time_str = spike_info['time_str']
    
    # Calculate sun positions
    poly_center_lat = (bounds[1] + bounds[3]) / 2
    poly_center_lon = (bounds[0] + bounds[2]) / 2
    
    normal_sun_az, normal_sun_el = calculate_sun_position(
        normal_info['time_dt'].to_pydatetime() if hasattr(normal_info['time_dt'], 'to_pydatetime') else normal_info['time_dt'],
        poly_center_lat, poly_center_lon
    )
    spike_sun_az, spike_sun_el = calculate_sun_position(
        spike_info['time_dt'].to_pydatetime() if hasattr(spike_info['time_dt'], 'to_pydatetime') else spike_info['time_dt'],
        poly_center_lat, poly_center_lon
    )
    
    normal_info['sun_azimuth'] = normal_sun_az
    normal_info['sun_elevation'] = normal_sun_el
    spike_info['sun_azimuth'] = spike_sun_az
    spike_info['sun_elevation'] = spike_sun_el
    
    # 1. Combined comparison (now 2x4 with sig0)
    plot_combined_comparison(normal_data, spike_data, normal_info, spike_info, polygon, bounds, output_dir, time_str, fctimp_str)
    
    # 2. Height comparison
    plot_height_comparison(normal_data, spike_data, normal_info, spike_info, polygon, bounds, output_dir, time_str, fctimp_str)
    
    # 3. Classification comparison
    plot_classification_comparison(normal_data, spike_data, normal_info, spike_info, polygon, bounds, output_dir, time_str, fctimp_str)
    
    # 4. Sig0 comparison (NEW)
    plot_sig0_comparison(normal_data, spike_data, normal_info, spike_info, polygon, bounds, output_dir, time_str, fctimp_str)
    
    # 5. Statistics comparison
    plot_statistics_comparison(normal_data, spike_data, normal_info, spike_info, output_dir, time_str, fctimp_str)
    
    print(f"  Saved: {time_str}")

def plot_combined_comparison(normal_data, spike_data, normal_info, spike_info, polygon, bounds, output_dir, time_str, fctimp_str):
    """2x4 grid: Top row = Normal, Bottom row = Spike (Height, Classification, Sig0, NLCD)"""
    
    dx = bounds[2] - bounds[0]
    dy = bounds[3] - bounds[1]
    aspect = dy / dx
    
    panel_width = 2.5
    fig_width = panel_width * 4 + 1.2  # 4 columns now
    fig_height = panel_width * aspect * 2 + 0.5
    
    fig, axes = plt.subplots(2, 4, figsize=(fig_width, fig_height), constrained_layout=True)
    
    nlcd_path = '/work/pi_kandread_umass_edu/swot-urban/0_data_collection/1_nlcd/Annual_NLCD_LndCov_2024_CU_C1V1_EPSG4326.tif'
    nlcd_colors = {11: '#476BA1', 12: '#D1DDF9', 21: '#DDC9C9', 22: '#D89382', 23: '#ED0000', 24: '#AA0000',
                   31: '#B2ADA3', 41: '#68AA63', 42: '#1C6330', 43: '#B5C98E', 52: '#CCBA7C', 71: '#E2E2C1',
                   81: '#DDD83E', 82: '#AE7229', 90: '#BBD7ED', 95: '#71A4C1'}
    
    for row, (data, info, label) in enumerate([
        (normal_data, normal_info, 'NORMAL'),
        (spike_data, spike_info, 'SPIKE')
    ]):
        unsel = ~data['selected_mask']
        sel = data['selected_mask']
        heading = info['heading']
        tile = info['tile']
        error = info['error']
        sun_az = info['sun_azimuth']
        sun_el = info['sun_elevation']
        
        # Column 0: Height
        ax = axes[row, 0]
        if sel.sum() > 0:
            ax.scatter(data['lon'][sel], data['lat'][sel], c=data['height'][sel],
                       cmap='viridis', s=0.15, alpha=0.7, marker='x', linewidths=0.15)
        if unsel.sum() > 0:
            ax.scatter(data['lon'][unsel], data['lat'][unsel], c=data['height'][unsel],
                       cmap='viridis', s=0.15, alpha=0.4, marker='o', edgecolors='none')

        polygon.boundary.plot(ax=ax, color=POLYGON_COLOR, linewidth=2)
        ax.set_title(f'{label} | {heading:.0f}°-{tile}\nError: {error:.2f}m', fontsize=10, fontweight='bold',
                     color='green' if label == 'NORMAL' else 'red')
        
        # Column 1: Classification
        ax = axes[row, 1]
        if sel.sum() > 0:
            ax.scatter(data['lon'][sel], data['lat'][sel], c=data['classification'][sel],
                       cmap=ListedColormap(CLASSIFICATION_COLORS), s=0.15, alpha=0.7, marker='x', linewidths=0.15, vmin=1, vmax=7)
        if unsel.sum() > 0:
            ax.scatter(data['lon'][unsel], data['lat'][unsel], c=data['classification'][unsel],
                       cmap=ListedColormap(CLASSIFICATION_COLORS), s=0.15, alpha=0.4, marker='o', vmin=1, vmax=7, edgecolors='none')

        polygon.boundary.plot(ax=ax, color=POLYGON_COLOR, linewidth=2)
        ax.set_title('Classification', fontsize=10)
        
        # Column 2: Sig0 (NEW)
        ax = axes[row, 2]
        if sel.sum() > 0:
            ax.scatter(data['lon'][sel], data['lat'][sel], c=data['sig0'][sel],
                       cmap='cool', s=0.15, alpha=0.7, marker='x', linewidths=0.15)
        if unsel.sum() > 0:
            ax.scatter(data['lon'][unsel], data['lat'][unsel], c=data['sig0'][unsel],
                       cmap='cool', s=0.15, alpha=0.4, marker='o', edgecolors='none')

        polygon.boundary.plot(ax=ax, color=POLYGON_COLOR, linewidth=2)
        ax.set_title('Sig0 (σ₀)', fontsize=10)
        
        # Column 3: NLCD
        ax = axes[row, 3]
        buffer = 0.005
        with rasterio.open(nlcd_path) as src:
            window = src.window(bounds[0]-buffer, bounds[1]-buffer, bounds[2]+buffer, bounds[3]+buffer)
            nlcd_data = src.read(1, window=window)
            unique_vals = np.unique(nlcd_data[nlcd_data > 0])
            cmap_list = [nlcd_colors.get(v, '#FFFFFF') for v in range(int(unique_vals.min()), int(unique_vals.max())+1)]
            extent = [bounds[0]-buffer, bounds[2]+buffer, bounds[1]-buffer, bounds[3]+buffer]
            ax.imshow(nlcd_data, extent=extent, cmap=ListedColormap(cmap_list),
                      vmin=unique_vals.min(), vmax=unique_vals.max(), interpolation='nearest')
        polygon.boundary.plot(ax=ax, color=POLYGON_COLOR, linewidth=2)
        ax.set_title('NLCD', fontsize=10)
        
        # Common settings
        for col, ax in enumerate(axes[row]):
            ax.set_xlim(bounds[0], bounds[2])
            ax.set_ylim(bounds[1], bounds[3])
            ax.set_aspect('equal')
            ax.axis('off')
            # Only add arrows and satellite info to NLCD panel (column 3)
            if col == 3:
                add_arrows_and_indicators(ax, bounds, heading, sun_az, sun_el, info['sat_info'])
    
    fig.suptitle(f'ISA={fctimp_str} | Normal vs Spike Comparison\n(X=selected, O=unselected)', 
                 fontsize=11, fontweight='bold')
    plt.savefig(f'{output_dir}/COMPARE_{time_str}_Combined.png', dpi=1200, bbox_inches='tight', pad_inches=0.15)
    plt.close()

def plot_height_comparison(normal_data, spike_data, normal_info, spike_info, polygon, bounds, output_dir, time_str, fctimp_str):
    """Side-by-side height comparison"""
    
    dx = bounds[2] - bounds[0]
    dy = bounds[3] - bounds[1]
    aspect = dy / dx
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 6 * aspect))
    
    for ax, (data, info, label) in zip(axes, [
        (normal_data, normal_info, 'NORMAL'),
        (spike_data, spike_info, 'SPIKE')
    ]):
        unsel = ~data['selected_mask']
        sel = data['selected_mask']
        
        if sel.sum() > 0:
            sc = ax.scatter(data['lon'][sel], data['lat'][sel], c=data['height'][sel],
                           cmap='viridis', s=0.15, alpha=0.7, marker='x', linewidths=0.15)
            plt.colorbar(sc, ax=ax, label='Height (m)', shrink=0.7)
            
        if unsel.sum() > 0:
            ax.scatter(data['lon'][unsel], data['lat'][unsel], c=data['height'][unsel],
                       cmap='viridis', s=0.15, alpha=0.4, marker='o', edgecolors='none')
        
        polygon.boundary.plot(ax=ax, color=POLYGON_COLOR, linewidth=2)
        ax.set_xlim(bounds[0], bounds[2])
        ax.set_ylim(bounds[1], bounds[3])
        ax.set_aspect('equal')
        ax.axis('off')
        ax.set_title(f'{label} | {info["heading"]:.0f}°-{info["tile"]}\nError: {info["error"]:.2f}m', 
                     fontsize=11, fontweight='bold', color='green' if label == 'NORMAL' else 'red')
    
    fig.suptitle(f'ISA={fctimp_str} | Height Comparison', fontsize=12, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.savefig(f'{output_dir}/COMPARE_{time_str}_Height.png', dpi=600, bbox_inches='tight')
    plt.close()

def plot_classification_comparison(normal_data, spike_data, normal_info, spike_info, polygon, bounds, output_dir, time_str, fctimp_str):
    """Side-by-side classification comparison"""
    
    dx = bounds[2] - bounds[0]
    dy = bounds[3] - bounds[1]
    aspect = dy / dx
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6 * aspect))
    
    for ax, (data, info, label) in zip(axes, [
        (normal_data, normal_info, 'NORMAL'),
        (spike_data, spike_info, 'SPIKE')
    ]):
        unsel = ~data['selected_mask']
        sel = data['selected_mask']
        if sel.sum() > 0:
            ax.scatter(data['lon'][sel], data['lat'][sel], c=data['classification'][sel],
                       cmap=ListedColormap(CLASSIFICATION_COLORS), s=0.15, alpha=0.7, marker='x', linewidths=0.15, vmin=1, vmax=7)
            
        if unsel.sum() > 0:
            ax.scatter(data['lon'][unsel], data['lat'][unsel], c=data['classification'][unsel],
                       cmap=ListedColormap(CLASSIFICATION_COLORS), s=0.15, alpha=0.4, marker='o', vmin=1, vmax=7, edgecolors='none')

        
        polygon.boundary.plot(ax=ax, color=POLYGON_COLOR, linewidth=2)
        ax.set_xlim(bounds[0], bounds[2])
        ax.set_ylim(bounds[1], bounds[3])
        ax.set_aspect('equal')
        ax.axis('off')
        ax.set_title(f'{label} | {info["heading"]:.0f}°-{info["tile"]}\nError: {info["error"]:.2f}m', 
                     fontsize=11, fontweight='bold', color='green' if label == 'NORMAL' else 'red')
    
    legend = [Patch(facecolor=CLASSIFICATION_COLORS[i], label=CLASSIFICATION_LABELS[i]) for i in range(7)]
    fig.legend(handles=legend, loc='center right', bbox_to_anchor=(0.99, 0.5), fontsize=9)
    
    fig.suptitle(f'ISA={fctimp_str} | Classification Comparison', fontsize=12, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 0.88, 0.94])
    plt.savefig(f'{output_dir}/COMPARE_{time_str}_Classification.png', dpi=600, bbox_inches='tight')
    plt.close()

def plot_sig0_comparison(normal_data, spike_data, normal_info, spike_info, polygon, bounds, output_dir, time_str, fctimp_str):
    """Side-by-side sig0 (radar backscatter) comparison"""
    
    dx = bounds[2] - bounds[0]
    dy = bounds[3] - bounds[1]
    aspect = dy / dx
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 6 * aspect))
    
    # Get common vmin/vmax for both plots
    all_sig0 = np.concatenate([
        normal_data['sig0'][normal_data['selected_mask']],
        spike_data['sig0'][spike_data['selected_mask']]
    ])
    # Filter out invalid values
    valid_sig0 = all_sig0[np.isfinite(all_sig0)]
    if len(valid_sig0) > 0:
        vmin = np.percentile(valid_sig0, 2)
        vmax = np.percentile(valid_sig0, 98)
    else:
        vmin, vmax = None, None
    
    for ax, (data, info, label) in zip(axes, [
        (normal_data, normal_info, 'NORMAL'),
        (spike_data, spike_info, 'SPIKE')
    ]):
        unsel = ~data['selected_mask']
        sel = data['selected_mask']
        if sel.sum() > 0:
            sc = ax.scatter(data['lon'][sel], data['lat'][sel], c=data['sig0'][sel],
                           cmap='cool', s=0.15, alpha=0.7, marker='x', linewidths=0.15,
                           vmin=vmin, vmax=vmax)
            plt.colorbar(sc, ax=ax, label='Sig0 (dB)', shrink=0.7)
            
        if unsel.sum() > 0:
            ax.scatter(data['lon'][unsel], data['lat'][unsel], c=data['sig0'][unsel],
                       cmap='cool', s=0.15, alpha=0.4, marker='o', edgecolors='none',
                       vmin=vmin, vmax=vmax)

        
        polygon.boundary.plot(ax=ax, color=POLYGON_COLOR, linewidth=2)
        ax.set_xlim(bounds[0], bounds[2])
        ax.set_ylim(bounds[1], bounds[3])
        ax.set_aspect('equal')
        ax.axis('off')
        ax.set_title(f'{label} | {info["heading"]:.0f}°-{info["tile"]}\nError: {info["error"]:.2f}m', 
                     fontsize=11, fontweight='bold', color='green' if label == 'NORMAL' else 'red')
    
    fig.suptitle(f'ISA={fctimp_str} | Sig0 (σ₀) Comparison', fontsize=12, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.savefig(f'{output_dir}/COMPARE_{time_str}_Sig0.png', dpi=600, bbox_inches='tight')
    plt.close()

def plot_statistics_comparison(normal_data, spike_data, normal_info, spike_info, output_dir, time_str, fctimp_str):
    """Side-by-side statistics comparison"""
    
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    
    for row, (data, info, label) in enumerate([
        (normal_data, normal_info, 'NORMAL'),
        (spike_data, spike_info, 'SPIKE')
    ]):
        sel = data['selected_mask']
        color = 'green' if label == 'NORMAL' else 'red'
        
        # Classification distribution
        axes[row, 0].bar(range(1, 8), [np.sum(data['classification'][sel] == i) for i in range(1, 8)], color=color, alpha=0.7)
        axes[row, 0].set_xlabel('Classification')
        axes[row, 0].set_ylabel('Count')
        axes[row, 0].set_title(f'{label}: Classification')
        axes[row, 0].grid(alpha=0.3)
        
        # Bright land
        axes[row, 1].bar(range(2), [np.sum(data['bright_land'][sel] == i) for i in range(2)], color=color, alpha=0.7)
        axes[row, 1].set_xlabel('Bright Land')
        axes[row, 1].set_ylabel('Count')
        axes[row, 1].set_title(f'{label}: Bright Land')
        axes[row, 1].set_xticks([0, 1])
        axes[row, 1].grid(alpha=0.3)
        
        # Boxplots
        if sel.sum() > 0:
            for col, key in zip([2, 3, 4], ['layover', 'sig0', 'coh_power']):
                bp = axes[row, col].boxplot(data[key][sel], vert=True, patch_artist=True)
                bp['boxes'][0].set_facecolor(color)
                bp['boxes'][0].set_alpha(0.5)
                axes[row, col].set_ylabel(key.replace('_', ' ').title())
                axes[row, col].set_title(f'{label}: {key.replace("_", " ").title()}')
                axes[row, col].grid(alpha=0.3)
    
    fig.suptitle(f'ISA={fctimp_str} | Statistics Comparison\nNormal (Error={normal_info["error"]:.2f}m) vs Spike (Error={spike_info["error"]:.2f}m)', 
                 fontsize=12, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.savefig(f'{output_dir}/COMPARE_{time_str}_Statistics.png', dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
# ============================================================
# FULL
# ============================================================
for fc in ['0.01', '0.06', '0.12', '0.17', '0.22', '0.32', '0.28', '0.40', '0.45', '0.50', '0.55', '0.61']:
    process_fctimp(fc, spike_threshold=0.1832)